# 07 · Haiyan STL scenarios and impact by spatial scale

Notebook 06 characterizes the pre-Haiyan baseline. Here, observability is checked
first, pre-event STL is fitted for an exploratory business-as-usual scenario, and
post-Haiyan departures are interpreted by spatial support. The regional G3
overview precedes municipalities, 5×5 areas, GHSL classes, and selected 1×1
pixels. Missing observed blocks stay missing; STL interpolation is confined to
model fitting.

## 1. Shared data, reference support, and profiles

In [ ]:
from pathlib import Path
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from rasterio.enums import Resampling
from rasterio.features import rasterize
from statsmodels.tsa.seasonal import STL
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import qualitative
from tqdm.auto import tqdm
from IPython.display import display

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (PROJECT_DIR / "datasets").exists():
    PROJECT_DIR = Path(
        "/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/"
        "02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery"
    )
DATA_DIR = PROJECT_DIR / "datasets"
PROCESSED_DIR = DATA_DIR / "VNP46/processed"
BASELINE_STORE = PROCESSED_DIR / "baseline/Haiyan_VNP46A2_baseline.zarr"
EXISTING_STORE = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"
MUNICIPALITIES_PATH = DATA_DIR / "boundaries/MuniCities/MuniCities.shp"
GHSL_PATH = next(DATA_DIR.glob("**/GHSL_SMOD_E2015.tif"))
OUTPUT_DIR = PROJECT_DIR / "output/notebook_07_four_scale_stl"
FIGURE_DIR, TABLE_DIR, CACHE_DIR = (OUTPUT_DIR / p for p in ("figures", "tables", "cache"))
for folder in (FIGURE_DIR, TABLE_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

EVENT_DATE = pd.Timestamp("2013-11-08")
REFERENCE_START = EVENT_DATE - pd.Timedelta(days=60)
PRE_EVENT_END = EVENT_DATE - pd.Timedelta(days=1)
AGGREGATION_DAYS = 4
SPATIAL_COMPLETENESS_PCT = 10.0
RQ_CLIP_PERCENTILE = 95.0
MIN_REFERENCE_BLOCKS = 3
FORECAST_DAYS = 365
PRIMARY_IMPACT_DAYS = 180
STL_PERIOD_BLOCKS = 7  # 28-day diagnostic, not an annual cycle
MIN_STL_OBSERVED_BLOCKS = 24
TREND_LOOKBACK_DAYS = 180
KNOWN_FILL_VALUES = (-9999., -999., -32768., 6553.5, 65535.)
REBUILD_PROFILES = False
DNB_BAND, MQF_BAND = "DNB_BRDF_Corrected_NTL", "Mandatory_Quality_Flag"

STUDY_AREAS = [
    "Tacloban", "Ormoc", "Baybay", "Catbalogan", "Borongan", "Guiuan",
    "Palo", "Tanauan", "Tolosa", "Dulag", "Basey", "Lawaan",
]
STUDY_PROVINCES = dict(zip(STUDY_AREAS, [
    "Leyte", "Leyte", "Leyte", "Samar", "Eastern Samar", "Eastern Samar",
    "Leyte", "Leyte", "Leyte", "Leyte", "Samar", "Eastern Samar",
]))
REGION_PROVINCES = ("Samar", "Eastern Samar", "Northern Samar", "Leyte", "Southern Leyte")
GHSL_CLASS_LABELS = {
    11: "Very low density rural", 12: "Low density rural", 13: "Rural cluster",
    21: "Suburban / peri-urban", 22: "Semi-dense urban", 23: "Dense urban",
    30: "Urban centre",
}
GHSL_CLASS_COLORS = {
    11: "#cdf57a", 12: "#abcd66", 13: "#375623", 21: "#ffff00",
    22: "#a87000", 23: "#732600", 30: "#ff0000",
}
POI_SPECS = [
    ("Airport", 125.02645, 11.22747, "#0072B2"),
    ("Port", 124.99865, 11.24905, "#E69F00"),
    ("Main-road vicinity", 125.00300, 11.24200, "#009E73"),
    ("BLISS Sagkahan residential", 125.00264, 11.21767, "#CC79A7"),
    ("Kassel City residential", 124.98314, 11.23742, "#7B5EA7"),
    ("Rizal Plaza", 125.00614, 11.24208, "#D55E00"),
    ("Astrodome", 125.00502, 11.22165, "#56B4E9"),
    ("Robinsons Place", 125.00778, 11.20752, "#B33D6A"),
]
CENTRE_POIS = [
    ("Tacloban", 125.0015, 11.2434, "Leyte"),
    ("Palo", 124.9904, 11.1577, "Leyte"),
    ("Guiuan", 125.7232, 11.0312, "Eastern Samar"),
    ("Alangalang", 124.8465, 11.2071, "Leyte"),
    ("Ormoc", 124.6068, 11.0088, "Leyte"),
    ("Baybay", 124.7989, 10.6766, "Leyte"),
]
POI_BOUNDS = (124.91, 11.13, 125.10, 11.33)
PLOT_FONT, PLOT_TEXT_COLOR, PLOT_GRID_COLOR = "Arial", "#243B5A", "#E8EDF3"
EVENT_COLOR, BASELINE_COLOR = "#0057FF", "#6C7882"
RQ_COLOR, RAW_COLOR = "#009227", "#0091FF"
AREA_COLORS = dict(zip(STUDY_AREAS, qualitative.Dark24))

def style_figure(fig, width=1450, height=780, bottom=115):
    fig.update_layout(
        template="plotly_white", paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white", width=width, height=height,
        font=dict(family=PLOT_FONT, size=14, color=PLOT_TEXT_COLOR),
        margin=dict(l=90, r=45, t=65, b=bottom),
        legend=dict(orientation="h", x=0, y=-.12, xanchor="left", yanchor="top",
                    font=dict(size=15)),
    )
    fig.update_xaxes(zeroline=False, gridcolor=PLOT_GRID_COLOR)
    fig.update_yaxes(zeroline=False, gridcolor=PLOT_GRID_COLOR)
    fig.update_annotations(font=dict(size=16, color=PLOT_TEXT_COLOR))
    return fig

def finish_figure(fig, filename, show=True):
    fig.write_html(FIGURE_DIR / (filename + ".html"), include_plotlyjs="directory")
    if show:
        fig.show()

In [ ]:
for path in (BASELINE_STORE, EXISTING_STORE, MUNICIPALITIES_PATH, GHSL_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

early = xr.open_zarr(BASELINE_STORE, chunks={}, consolidated=False, mask_and_scale=False)
existing = xr.open_zarr(EXISTING_STORE, chunks={}, consolidated=False, mask_and_scale=False)
if not np.all(early.processed.values == 1):
    raise RuntimeError("Finish Notebook 05's baseline store before running this notebook.")
for axis in ("x", "y"):
    np.testing.assert_allclose(early[axis].values, existing[axis].values, rtol=0, atol=1e-7)
early = early.assign_coords(x=existing.x, y=existing.y)
early["spatial_ref"] = existing["spatial_ref"]
existing = existing.isel(date=np.flatnonzero(existing.processed.values == 1))
new_dates = pd.DatetimeIndex(early.date.values).difference(existing.date.values)
a2 = xr.concat(
    [early.sel(date=new_dates), existing], dim="date", data_vars="minimal",
    coords="minimal", compat="equals", join="exact",
).sortby("date")
a2 = a2.rio.write_crs(existing.attrs["crs_wkt"])
dates = pd.DatetimeIndex(a2.date.values)
if not dates.is_unique:
    raise ValueError("Duplicate dates remain after combining the A2 stores.")
day_blocks = np.asarray((dates - EVENT_DATE).days // AGGREGATION_DAYS)
all_blocks = np.arange(day_blocks.min(), day_blocks.max() + 1)
print(f"A2 coverage: {len(dates):,} available dates, {dates.min():%Y-%m-%d} to {dates.max():%Y-%m-%d}.")

municipalities = gpd.read_file(MUNICIPALITIES_PATH)
name_col = next(c for c in ("ADM3_EN", "NAME_2") if c in municipalities.columns)
province_col = next(c for c in ("ADM2_EN", "PROVINCE") if c in municipalities.columns)
municipalities["unit_name"] = (
    municipalities[name_col].astype(str).str.upper()
    .str.replace(r"^CITY OF\s+", "", regex=True)
    .str.replace(r"\s+CITY$", "", regex=True).str.strip().str.title()
)
municipalities["province_name"] = municipalities[province_col].astype(str).str.strip().str.title()
region = municipalities[
    municipalities.province_name.isin(REGION_PROVINCES)
].to_crs(a2.rio.crs).copy()
focused = region[
    region.unit_name.isin(STUDY_AREAS)
    & region.province_name.eq(region.unit_name.map(STUDY_PROVINCES))
].copy()
counts = focused.unit_name.value_counts().reindex(STUDY_AREAS, fill_value=0)
if not counts.eq(1).all():
    raise ValueError(f"Expected one boundary per Notebook 04c area: {counts.to_dict()}")
focused = focused.set_index("unit_name").loc[STUDY_AREAS].reset_index()
focused["profile_id"] = np.arange(1, len(focused) + 1)

template = a2[DNB_BAND].isel(date=0, drop=True)
ghsl = rxr.open_rasterio(GHSL_PATH, masked=True).squeeze("band", drop=True)
ghsl = ghsl.rio.reproject_match(template, resampling=Resampling.nearest).values
transform = template.rio.transform(recalc=True)
regional_zone = rasterize(
    [(geom, 1) for geom in region.geometry], out_shape=ghsl.shape,
    transform=transform, fill=0, all_touched=False,
)
municipal_zone = rasterize(
    [(geom, int(pid)) for geom, pid in zip(focused.geometry, focused.profile_id)],
    out_shape=ghsl.shape, transform=transform, fill=0, all_touched=False,
)
land = (regional_zone > 0) & np.isin(ghsl, list(GHSL_CLASS_LABELS))
land_pixels = np.flatnonzero(land)
ghsl_vector = ghsl.ravel()[land_pixels].astype(int)
g3_vector = np.isin(ghsl_vector, (22, 23, 30))
municipal_vector = municipal_zone.ravel()[land_pixels]
print(f"Samar–Leyte land support: {len(land_pixels):,} pixels; "
      f"focused municipalities: {len(focused)}.")

In [ ]:
def nanmedian(values, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmedian(values, axis=axis)

def read_days(indices):
    batch = a2[[DNB_BAND, MQF_BAND]].isel(date=indices).compute()
    raw = batch[DNB_BAND].values.reshape(len(indices), -1)[:, land_pixels].astype("float32")
    quality = batch[MQF_BAND].values.reshape(len(indices), -1)[:, land_pixels]
    raw[(quality != 0) | ~np.isfinite(raw) | (raw < 0)
        | np.isin(raw, KNOWN_FILL_VALUES)] = np.nan
    capped = raw.copy()
    for j in range(len(indices)):
        good = raw[j, g3_vector]
        good = good[np.isfinite(good)]
        if good.size:
            capped[j] = np.minimum(raw[j], np.percentile(good, RQ_CLIP_PERCENTILE))
    return capped

cache_tag = f"{len(dates)}_{dates.min():%Y%m%d}_{dates.max():%Y%m%d}_p95"
reference_path = CACHE_DIR / f"reference_{cache_tag}.npz"
if reference_path.exists() and not REBUILD_PROFILES:
    saved = np.load(reference_path)
    ntl0, reference_count = saved["ntl0"], saved["reference_count"]
else:
    reference_composites = []
    for block in tqdm(range(-15, 0), desc="60-day reference"):
        indices = np.flatnonzero(day_blocks == block)
        reference_composites.append(
            nanmedian(read_days(indices), axis=0) if len(indices)
            else np.full(len(land_pixels), np.nan, dtype="float32")
        )
    reference_composites = np.asarray(reference_composites)
    reference_count = np.isfinite(reference_composites).sum(axis=0)
    ntl0 = nanmedian(reference_composites, axis=0)
    np.savez_compressed(reference_path, ntl0=ntl0, reference_count=reference_count)
reference_ok = (reference_count >= MIN_REFERENCE_BLOCKS) & np.isfinite(ntl0) & (ntl0 > 0)

profiles, meta = {}, []
def add_profile(pid, label, scale, indices, color, municipality="", ghsl_code=np.nan,
                longitude=np.nan, latitude=np.nan, snap_km=np.nan):
    indices = np.asarray(indices, dtype=int)
    profiles[pid] = indices
    meta.append(dict(
        profile_id=pid, label=label, scale=scale, color=color,
        municipality=municipality, ghsl_code=ghsl_code,
        longitude=longitude, latitude=latitude, snap_km=snap_km,
        base_pixels=len(indices), fixed_pixels=int(reference_ok[indices].sum()),
    ))

for row in focused.itertuples():
    support = np.flatnonzero((municipal_vector == row.profile_id) & g3_vector)
    color = AREA_COLORS[row.unit_name]
    add_profile(f"municipality::{row.unit_name}", row.unit_name,
                "municipality", support, color, municipality=row.unit_name)
    eligible = support[reference_ok[support]]
    if not len(eligible):
        print(f"{row.unit_name}: no reference-lit G3 anchor for a 5×5 window.")
        continue
    anchor = eligible[np.argmax(ntl0[eligible])]
    iy, ix = np.unravel_index(land_pixels[anchor], land.shape)
    window = np.zeros(land.shape, dtype=bool)
    window[max(0, iy-2):iy+3, max(0, ix-2):ix+3] = True
    local = np.flatnonzero(window.ravel()[land_pixels]
                            & (municipal_vector == row.profile_id) & g3_vector)
    add_profile(f"kernel5::{row.unit_name}", row.unit_name + " · bright 5×5",
                "kernel5", local, color, municipality=row.unit_name,
                longitude=float(a2.x.values[ix]), latitude=float(a2.y.values[iy]))

for name, lon, lat, province in CENTRE_POIS:
    parent = region[region.unit_name.eq(name) & region.province_name.eq(province)]
    if len(parent) != 1:
        print(f"{name}: centre POI has no unique municipal boundary.")
        continue
    ix = int(np.abs(a2.x.values - lon).argmin())
    iy = int(np.abs(a2.y.values - lat).argmin())
    window = np.zeros(land.shape, dtype=bool)
    window[max(0, iy-2):iy+3, max(0, ix-2):ix+3] = True
    parent_mask = rasterize([(parent.geometry.iloc[0], 1)], out_shape=land.shape,
                            transform=transform, fill=0, all_touched=False).astype(bool)
    local = np.flatnonzero((window & parent_mask).ravel()[land_pixels] & g3_vector)
    add_profile(f"centre5::{name}", name + " · centre 5×5", "kernel5",
                local, AREA_COLORS.get(name, "#174A7E"), municipality=name,
                longitude=float(a2.x.values[ix]), latitude=float(a2.y.values[iy]))

for ghsl_code, label in GHSL_CLASS_LABELS.items():
    add_profile(f"ghsl::{ghsl_code}", f"{ghsl_code} · {label}", "ghsl",
                np.flatnonzero(ghsl_vector == ghsl_code), GHSL_CLASS_COLORS[ghsl_code],
                ghsl_code=ghsl_code)

eligible = np.flatnonzero(reference_ok)
yy, xx = np.unravel_index(land_pixels[eligible], land.shape)
candidate = pd.DataFrame({
    "index": eligible, "ix": xx, "iy": yy,
    "x": a2.x.values[xx], "y": a2.y.values[yy],
    "class_code": ghsl_vector[eligible],
})
candidate = candidate[
    candidate.x.between(POI_BOUNDS[0], POI_BOUNDS[2])
    & candidate.y.between(POI_BOUNDS[1], POI_BOUNDS[3])
].copy()
used_pixels = set()
for name, lon, lat, color in POI_SPECS:
    available = candidate[~candidate.index.isin(used_pixels)]
    if available.empty:
        break
    km = 111.2 * np.hypot((available.x - lon) * np.cos(np.deg2rad(lat)), available.y - lat)
    chosen = available.loc[km.idxmin()]
    shift_km = float(km.min())
    if shift_km > 1.5:
        print(f"Skipped {name}: nearest eligible pixel is {shift_km:.2f} km away.")
        continue
    used_pixels.add(chosen.name)
    add_profile(f"pixel1::{name}", name, "pixel1", [int(chosen["index"])], color,
                ghsl_code=int(chosen.class_code), longitude=float(chosen.x),
                latitude=float(chosen.y), snap_km=shift_km)

for ghsl_code in (23, 22, 21, 13, 12, 11):
    available = candidate[(candidate.class_code == ghsl_code)
                          & ~candidate.index.isin(used_pixels)]
    if available.empty:
        print(f"No eligible 1×1 GHSL {ghsl_code} pixel in the Tacloban view.")
        continue
    km = 111.2 * np.hypot((available.x - 125.0015) * np.cos(np.deg2rad(11.2434)),
                          available.y - 11.2434)
    chosen = available.loc[km.idxmin()]
    used_pixels.add(chosen.name)
    add_profile(f"pixel1::Pixel {ghsl_code}", f"Pixel {ghsl_code}", "pixel1",
                [int(chosen["index"])], GHSL_CLASS_COLORS[ghsl_code],
                ghsl_code=ghsl_code, longitude=float(chosen.x), latitude=float(chosen.y))

profile_meta = pd.DataFrame(meta)
profile_meta["fixed_support_share"] = (
    profile_meta.fixed_pixels / profile_meta.base_pixels.replace(0, np.nan)
)
fixed_profiles = {pid: idx[reference_ok[idx]] for pid, idx in profiles.items()}
display(profile_meta.groupby("scale").agg(
    profiles=("profile_id", "size"),
    fixed_pixels=("fixed_pixels", "sum"),
).reindex(["municipality", "kernel5", "ghsl", "pixel1"]))
display(profile_meta.loc[profile_meta.scale.eq("pixel1"),
                         ["label", "ghsl_code", "longitude", "latitude", "snap_km"]].round(3))

In [ ]:
# One regional G3 overview profile, using the existing support and reference rules.
regional_id = "regional::G3"
if regional_id not in profiles:
    regional_support = np.flatnonzero(g3_vector)
    add_profile(regional_id, "Regional Samar–Leyte G3", "regional",
                regional_support, RQ_COLOR)
    profile_meta = pd.DataFrame(meta)
    profile_meta["fixed_support_share"] = (
        profile_meta.fixed_pixels / profile_meta.base_pixels.replace(0, np.nan)
    )
    fixed_profiles[regional_id] = regional_support[reference_ok[regional_support]]

In [ ]:
# ------------------------------------------------------------
# Display grids — analytical values remain unchanged
# ------------------------------------------------------------

map_classes = [
    (10, "Water", "#B8D9EF"),
    *[
        (value, GHSL_CLASS_LABELS[value], GHSL_CLASS_COLORS[value])
        for value in GHSL_CLASS_LABELS
    ],
]

ghsl_grid = np.full(land.shape, np.nan)

for index, (value, _, _) in enumerate(map_classes):
    mask = (ghsl == value)
    if value != 10:
        mask &= land
    ghsl_grid[mask] = index

reference_grid = np.full(land.size, np.nan)
reference_grid[land_pixels[reference_ok]] = ntl0[reference_ok]
reference_grid = reference_grid.reshape(land.shape)

# Water is shown only where GHSL identifies water.
# Missing terrestrial NTL remains blank.
water_grid = np.where(ghsl == 10, 1.0, np.nan)

ghsl_scale = [
    [stop, color]
    for index, (_, _, color) in enumerate(map_classes)
    for stop in (
        index / len(map_classes),
        (index + 1) / len(map_classes),
    )
]

west, south, east, north = region.total_bounds
regional_bounds = (
    west - 0.03,
    south - 0.03,
    east + 0.03,
    north + 0.03,
)
tacloban_bounds = (124.94, 11.14, 125.10, 11.32)

# Dissolve internal municipality boundaries.
outline = region.dissolve().geometry.iloc[0].boundary
outline_x, outline_y = [], []

for line in (
    list(outline.geoms)
    if hasattr(outline, "geoms")
    else [outline]
):
    xx, yy = line.xy
    outline_x.extend([*xx, None])
    outline_y.extend([*yy, None])



## 2. Four-day extraction and observability

The added regional overview uses the same pixel-quality, P95, reference-support, and SC rules as the four established scales.

In [ ]:
signature = hashlib.sha256()
signature.update(cache_tag.encode())
signature.update(str(SPATIAL_COMPLETENESS_PCT).encode())
for pid, fixed in fixed_profiles.items():
    signature.update(pid.encode())
    signature.update(np.asarray(fixed, dtype="int32").tobytes())
profile_cache = CACHE_DIR / ("four_day_" + signature.hexdigest()[:16] + ".csv")

if profile_cache.exists() and not REBUILD_PROFILES:
    four_day = pd.read_csv(profile_cache, parse_dates=["date", "date_start", "date_end"])
else:
    rows = []
    empty_composite = np.full(len(land_pixels), np.nan, dtype="float32")
    for block in tqdm(all_blocks, desc="Four-day profiles across all four scales"):
        indices = np.flatnonzero(day_blocks == block)
        composite = nanmedian(read_days(indices), axis=0) if len(indices) else empty_composite
        date_start = EVENT_DATE + pd.Timedelta(days=int(block) * AGGREGATION_DAYS)
        for pid, fixed in fixed_profiles.items():
            values = composite[fixed]
            valid = np.isfinite(values)
            n_valid, n_fixed = int(valid.sum()), len(fixed)
            sc_pct = 100 * n_valid / n_fixed if n_fixed else np.nan
            observed = n_valid > 0 and sc_pct >= SPATIAL_COMPLETENESS_PCT
            ntl = float(np.median(values[valid])) if observed else np.nan
            matched_pct = (
                100 * float(values[valid].sum()) / float(ntl0[fixed][valid].sum())
                if observed else np.nan
            )
            rows.append(dict(
                profile_id=pid, block=int(block),
                date_start=date_start,
                date_end=date_start + pd.Timedelta(days=AGGREGATION_DAYS - 1),
                date=date_start + pd.Timedelta(days=1.5),
                n_fixed=n_fixed, n_valid=n_valid, sc_pct=sc_pct,
                observed=observed, ntl_rq=ntl, matched_reference_pct=matched_pct,
            ))
    four_day = pd.DataFrame(rows)
    four_day.to_csv(profile_cache, index=False)

four_day["observed"] = four_day.observed.astype(bool)
four_day = four_day.merge(profile_meta[
    ["profile_id", "label", "scale", "color", "ghsl_code"]
], on="profile_id", how="left", validate="many_to_one")
four_day["status"] = np.where(four_day.observed, "observed", "not observable")
profile_meta.to_csv(TABLE_DIR / "profile_support.csv", index=False)
four_day.to_csv(TABLE_DIR / "four_day_profiles.csv", index=False)
print(f"Extracted {len(profile_meta)} profiles across {len(all_blocks)} calendar blocks; "
      f"last processed day: {dates.max():%Y-%m-%d}.")
display(four_day.groupby("scale").agg(
    profiles=("profile_id", "nunique"),
    median_observed_share=("observed", "mean"),
    median_sc_pct=("sc_pct", "median"),
).round(3))

The green heatmap shows when each spatial support was observable. Blank four-day blocks are retained in the calendar and must not be read as darkness.

In [ ]:
scale_order = ["municipality", "kernel5", "ghsl", "pixel1"]
scale_titles = [
    "Municipality-wide G3",
    "5×5 anchors and centres",
    "GHSL classes",
    "Selected 1×1 pixels",
]

sc_colorscale = [
    [0.00, "#F3F5F8"],
    [0.10, "#E6F3E8"],
    [0.35, "#A9D8B1"],
    [0.70, "#48A46F"],
    [1.00, "#00582B"],
]

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=.075,
    subplot_titles=scale_titles,
)

for row, scale in enumerate(scale_order, start=1):
    members = profile_meta.loc[
        profile_meta.scale.eq(scale), "profile_id"
    ].tolist()

    coverage = (
        four_day[four_day.scale.eq(scale)]
        .pivot(index="profile_id", columns="date", values="sc_pct")
        .reindex(members)
    )

    labels = profile_meta.set_index("profile_id").loc[members, "label"]

    axis_name = "yaxis" if row == 1 else f"yaxis{row}"
    y0, y1 = fig.layout[axis_name].domain

    fig.add_trace(
        go.Heatmap(
            x=coverage.columns,
            y=labels,
            z=coverage.values,
            colorscale=sc_colorscale,
            zmin=0,
            zmax=100,
            hoverongaps=False,
            colorbar=dict(
                title=dict(text="SC (%)", font=dict(size=13)),
                x=1.01,
                y=(y0 + y1) / 2,
                len=y1 - y0,
                thickness=14,
                tickvals=[0, 50, 100],
                tickfont=dict(size=11),
                outlinewidth=0,
            ),
            hovertemplate=(
                "%{y}<br>%{x|%d %b %Y}"
                "<br>Spatial completeness %{z:.1f}%"
                "<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=1,
    )

    fig.add_vline(
        x=EVENT_DATE,
        line_dash="dash",
        line_color=EVENT_COLOR,
        line_width=1.5,
        row=row, col=1,
    )
    fig.update_yaxes(tickfont=dict(size=11), row=row, col=1)

fig.update_xaxes(
    tickformat="%b %Y",
    dtick="M6",
    tickfont=dict(size=12),
    row=4, col=1,
)

style_figure(fig, width=1600, height=1450, bottom=85)
fig.update_layout(margin=dict(l=220, r=135, t=70, b=85))
finish_figure(fig, "01_observability_by_scale")

## 3. Shared STL fit, BAU scenario, and validation

The fit uses only pre-Haiyan data. Its regularized STL input is distinct from the observed RQ series, which remains gappy. The post-event curve is a scenario, not a causal counterfactual.

In [ ]:
pre_blocks = np.arange(all_blocks.min(), 0)
future_blocks = np.arange(0, int(np.ceil(FORECAST_DAYS / AGGREGATION_DAYS)))
stl_rows, forecast_rows, model_rows, lag_rows = [], [], [], []

for row in tqdm(profile_meta.itertuples(), total=len(profile_meta), desc="Pre-Haiyan STL"):
    g = four_day[four_day.profile_id.eq(row.profile_id)].set_index("block")
    observed = g.ntl_rq.reindex(pre_blocks)
    n_observed = int(observed.notna().sum())
    if n_observed < MIN_STL_OBSERVED_BLOCKS:
        model_rows.append(dict(profile_id=row.profile_id, fit_status="insufficient baseline observability",
                               baseline_observed_blocks=n_observed, baseline_calendar_blocks=len(pre_blocks)))
        continue

    filled = observed.interpolate(method="linear", limit_direction="both")
    missing = observed.isna()
    gap_id = missing.ne(missing.shift(fill_value=False)).cumsum()
    longest_gap = max((len(run) for _, run in missing.groupby(gap_id) if run.all()), default=0)
    result = STL(np.log1p(filled), period=STL_PERIOD_BLOCKS, robust=True).fit()
    trend = np.asarray(result.trend)
    seasonal = np.asarray(result.seasonal)
    residual = np.asarray(result.resid)

    # Recurrence check: pairwise correlation on genuinely observed, detrended blocks.
    detrended = np.log1p(observed.to_numpy(dtype=float)) - trend
    lag_checks = []
    for lag in list(range(5, 21)) + list(range(84, 99)):
        left, right = detrended[:-lag], detrended[lag:]
        paired = np.isfinite(left) & np.isfinite(right)
        corr = (float(np.corrcoef(left[paired], right[paired])[0, 1])
                if paired.sum() >= 20 and np.std(left[paired]) > 0
                and np.std(right[paired]) > 0 else np.nan)
        lag_checks.append((lag, int(paired.sum()), corr))
        lag_rows.append(dict(profile_id=row.profile_id, lag_days=lag * AGGREGATION_DAYS,
                             paired_blocks=int(paired.sum()), correlation=corr))
    eligible_lags = [(lag, pairs, corr) for lag, pairs, corr in lag_checks
                     if lag <= 20 and np.isfinite(corr)]
    best_lag, best_pairs, best_corr = (
        max(eligible_lags, key=lambda item: item[2]) if eligible_lags else (np.nan, 0, np.nan)
    )
    cycle_days = best_lag * AGGREGATION_DAYS if best_corr >= .2 else np.nan
    annual_lags = [(lag, pairs, corr) for lag, pairs, corr in lag_checks
                   if lag >= 84 and np.isfinite(corr)]
    annual_lag, annual_pairs, annual_corr = (
        max(annual_lags, key=lambda item: item[2]) if annual_lags else (np.nan, 0, np.nan)
    )
    seasonal_strength = max(0., 1. - np.var(residual) / np.var(seasonal + residual)) \
        if np.var(seasonal + residual) > 0 else np.nan

    recent = (pre_blocks >= -(TREND_LOOKBACK_DAYS // AGGREGATION_DAYS)) & observed.notna().to_numpy()
    recent_idx = np.flatnonzero(recent)
    slope = float(np.polyfit(pre_blocks[recent_idx], trend[recent_idx], 1)[0]) \
        if len(recent_idx) >= 10 else 0.
    slope_status = "estimated" if len(recent_idx) >= 10 else "flat fallback: sparse recent baseline"
    seasonal_template = np.array([
        np.median(seasonal[pre_blocks % STL_PERIOD_BLOCKS == phase])
        for phase in range(STL_PERIOD_BLOCKS)
    ])
    future_seasonal = seasonal_template[future_blocks % STL_PERIOD_BLOCKS]
    future_trend = trend[-1] + slope * (future_blocks + 1)
    expected_log = future_trend + future_seasonal
    expected = np.maximum(0., np.expm1(expected_log))
    expected_flat = np.maximum(0., np.expm1(trend[-1] + future_seasonal))
    observed_residual = residual[observed.notna().to_numpy()]
    lo, hi = np.quantile(observed_residual, [.10, .90])
    lower = np.maximum(0., np.expm1(expected_log + lo))
    upper = np.maximum(0., np.expm1(expected_log + hi))

    for block, obs, fill, tr, seas, resid in zip(pre_blocks, observed, filled,
                                                  trend, seasonal, residual):
        stl_rows.append(dict(profile_id=row.profile_id, block=int(block),
                             date=EVENT_DATE + pd.Timedelta(days=int(block) * 4 + 1.5),
                             observed_input=bool(np.isfinite(obs)),
                             observed_ntl=obs, stl_input=fill,
                             trend_log=tr, seasonal_log=seas, residual_log=resid,
                             fitted_ntl=max(0., np.expm1(tr + seas))))
    for block, value, flat, low, high, tr, seas in zip(
        future_blocks, expected, expected_flat, lower, upper, future_trend, future_seasonal
    ):
        forecast_rows.append(dict(profile_id=row.profile_id, block=int(block),
                                  date=EVENT_DATE + pd.Timedelta(days=int(block) * 4 + 1.5),
                                  expected_ntl=value, flat_trend_ntl=flat,
                                  variability_low=low, variability_high=high,
                                  projected_trend_log=tr, projected_seasonal_log=seas))
    model_rows.append(dict(
        profile_id=row.profile_id, fit_status="exploratory STL fitted",
        baseline_observed_blocks=n_observed, baseline_calendar_blocks=len(pre_blocks),
        baseline_interpolated_pct=100 * missing.mean(), longest_missing_gap_days=longest_gap * 4,
        recent_trend_blocks=len(recent_idx), trend_slope_log_per_4d=slope,
        trend_status=slope_status, stl_period_days=STL_PERIOD_BLOCKS * 4,
        observed_cycle_peak_days=cycle_days, peak_lag_pairs=best_pairs,
        peak_lag_correlation=best_corr, seasonal_strength=seasonal_strength,
        annual_lag_peak_days=annual_lag * AGGREGATION_DAYS,
        annual_lag_pairs=annual_pairs, annual_lag_correlation=annual_corr,
        seasonal_p10_log=np.quantile(seasonal, .10), seasonal_p90_log=np.quantile(seasonal, .90),
        residual_mad_log=np.median(np.abs(observed_residual - np.median(observed_residual))),
    ))

model_audit = pd.DataFrame(model_rows).merge(
    profile_meta[["profile_id", "label", "scale", "fixed_pixels"]], on="profile_id"
)
stl_components = pd.DataFrame(stl_rows)
forecast = pd.DataFrame(forecast_rows)
lag_table = pd.DataFrame(lag_rows)
model_audit.to_csv(TABLE_DIR / "stl_support_and_seasonality.csv", index=False)
stl_components.to_csv(TABLE_DIR / "pre_haiyan_stl_components.csv", index=False)
forecast.to_csv(TABLE_DIR / "business_as_usual_scenarios.csv", index=False)
lag_table.to_csv(TABLE_DIR / "observed_lag_correlations.csv", index=False)
display(model_audit.groupby("scale").agg(
    profiles=("profile_id", "size"), fitted=("fit_status", lambda s: s.eq("exploratory STL fitted").sum()),
    median_interpolated_pct=("baseline_interpolated_pct", "median"),
    median_seasonal_strength=("seasonal_strength", "median"),
).round(2))

In [ ]:
MIN_EXPECTED_FOR_PCT = .10
post = four_day[four_day.block.ge(0)].copy()
comparison = post.merge(
    forecast[["profile_id", "block", "expected_ntl", "flat_trend_ntl",
              "variability_low", "variability_high"]],
    on=["profile_id", "block"], how="inner", validate="one_to_one",
)
comparison["difference_ntl"] = comparison.ntl_rq - comparison.expected_ntl
comparison["difference_pct"] = np.where(
    comparison.observed & comparison.expected_ntl.ge(MIN_EXPECTED_FOR_PCT),
    100 * comparison.difference_ntl / comparison.expected_ntl, np.nan,
)
comparison["below_variability_band"] = (
    comparison.observed & comparison.ntl_rq.lt(comparison.variability_low)
)

impact_rows = []
for pid in profile_meta.profile_id:
    profile_comparison = comparison[comparison.profile_id.eq(pid)]
    for days in (60, 180, 365):
        g = profile_comparison[
            profile_comparison.date_start.lt(EVENT_DATE + pd.Timedelta(days=days))
        ]
        observed_g = g[g.observed & g.expected_ntl.notna()]
        pct_g = observed_g[observed_g.expected_ntl.ge(MIN_EXPECTED_FOR_PCT)]
        worst = pct_g.loc[pct_g.difference_pct.idxmin()] if len(pct_g) else None
        denominator = observed_g.expected_ntl.sum()
        impact_rows.append(dict(
            profile_id=pid, window_days=days, calendar_blocks=len(g),
            observed_blocks=len(observed_g),
            observable_share=len(observed_g) / len(g) if len(g) else np.nan,
            percent_comparable_blocks=len(pct_g),
            first_observed_date=observed_g.date_start.min() if len(observed_g) else pd.NaT,
            worst_date=worst.date_start if worst is not None else pd.NaT,
            worst_deviation_pct=worst.difference_pct if worst is not None else np.nan,
            worst_absolute_difference=observed_g.difference_ntl.min() if len(observed_g) else np.nan,
            median_deviation_pct=pct_g.difference_pct.median() if len(pct_g) else np.nan,
            observed_block_shortfall_pct=(
                100 * np.maximum(0, observed_g.expected_ntl - observed_g.ntl_rq).sum() / denominator
                if denominator > 0 else np.nan
            ),
            below_variability_band_blocks=int(observed_g.below_variability_band.sum()),
            below_80pct_expected_blocks=int(
                (pct_g.ntl_rq < .8 * pct_g.expected_ntl).sum()
            ),
        ))
impact = pd.DataFrame(impact_rows).merge(
    profile_meta[["profile_id", "label", "scale", "color", "fixed_pixels"]], on="profile_id"
).merge(model_audit[["profile_id", "fit_status", "baseline_interpolated_pct",
                     "longest_missing_gap_days", "seasonal_strength"]], on="profile_id")
impact["interpretability"] = np.select([
    impact.fit_status.ne("exploratory STL fitted"),
    impact.observed_blocks.lt(5) | impact.observable_share.lt(.20),
    impact.baseline_interpolated_pct.gt(30) | impact.longest_missing_gap_days.gt(60),
], ["no baseline model", "observation-limited", "baseline interpolation-sensitive"],
    default="descriptive comparison")

comparison.to_csv(TABLE_DIR / "observed_vs_business_as_usual.csv", index=False)
impact.to_csv(TABLE_DIR / "impact_metrics_by_window.csv", index=False)
display(impact[impact.window_days.eq(PRIMARY_IMPACT_DAYS)][[
    "scale", "label", "fixed_pixels", "observed_blocks", "observable_share",
    "worst_deviation_pct", "observed_block_shortfall_pct", "interpretability",
]].sort_values(["scale", "worst_deviation_pct"]).round(2))

The pre-Haiyan holdout checks whether the projected scenario predicts qualified observations better than a recent-median benchmark. Error estimates remain conditional on the selected reference pixels.

In [ ]:
holdout_start = -int(np.ceil(90 / AGGREGATION_DAYS))
train_blocks = pre_blocks[pre_blocks < holdout_start]
test_blocks = pre_blocks[pre_blocks >= holdout_start]
backtest_rows = []
for row in tqdm(profile_meta.itertuples(), total=len(profile_meta), desc="Baseline holdout"):
    g = four_day[four_day.profile_id.eq(row.profile_id)].set_index("block")
    observed = g.ntl_rq.reindex(pre_blocks)
    train = observed.reindex(train_blocks)
    test = observed.reindex(test_blocks)
    if train.notna().sum() < MIN_STL_OBSERVED_BLOCKS or test.notna().sum() < 5:
        backtest_rows.append(dict(profile_id=row.profile_id, holdout_blocks=int(test.notna().sum()),
                                  holdout_mae_ntl=np.nan, holdout_median_ape_pct=np.nan))
        continue
    fit = train.interpolate(method="linear", limit_direction="both")
    result = STL(np.log1p(fit), period=STL_PERIOD_BLOCKS, robust=True).fit()
    tr, seas = np.asarray(result.trend), np.asarray(result.seasonal)
    recent = (train_blocks >= train_blocks.max() - TREND_LOOKBACK_DAYS // 4) & train.notna().to_numpy()
    recent_idx = np.flatnonzero(recent)
    slope = float(np.polyfit(train_blocks[recent_idx], tr[recent_idx], 1)[0]) \
        if len(recent_idx) >= 10 else 0.
    phase = np.array([
        np.median(seas[train_blocks % STL_PERIOD_BLOCKS == j])
        for j in range(STL_PERIOD_BLOCKS)
    ])
    prediction = np.maximum(0., np.expm1(
        tr[-1] + slope * (test_blocks - train_blocks[-1])
        + phase[test_blocks % STL_PERIOD_BLOCKS]
    ))
    observed_test = test.to_numpy(dtype=float)
    valid = np.isfinite(observed_test)
    backtest_rows.append(dict(
        profile_id=row.profile_id, holdout_blocks=int(valid.sum()),
        holdout_mae_ntl=float(np.mean(np.abs(observed_test[valid] - prediction[valid]))),
        holdout_median_ape_pct=float(np.median(
            100 * np.abs(observed_test[valid] - prediction[valid])
            / np.maximum(prediction[valid], MIN_EXPECTED_FOR_PCT)
        )),
    ))
backtest = pd.DataFrame(backtest_rows).merge(
    profile_meta[["profile_id", "label", "scale"]], on="profile_id"
)
backtest.to_csv(TABLE_DIR / "pre_haiyan_holdout_check.csv", index=False)
display(backtest.groupby("scale").agg(
    tested=("holdout_mae_ntl", "count"),
    median_mae_ntl=("holdout_mae_ntl", "median"),
    median_ape_pct=("holdout_median_ape_pct", "median"),
).round(2))

The flat/calendar-median benchmark is retained separately. Its table has a distinct name so it cannot overwrite the STL scenario used for impact metrics.

In [ ]:
# BAU from observed pre-Haiyan blocks only.
# Calendar matching is used only when it beats a flat median in a pre-event holdout.

BAU_HOLDOUT_DAYS = 180
CALENDAR_WINDOW_DAYS = 30
MIN_CALENDAR_MATCHES = 6

def calendar_median(target_dates, reference, fallback):
    reference_days = pd.DatetimeIndex(reference.date).dayofyear.to_numpy()
    reference_ntl = reference.ntl_rq.to_numpy(dtype=float)

    predictions = np.full(len(target_dates), fallback, dtype=float)
    matches = np.zeros(len(target_dates), dtype=int)

    for i, date in enumerate(pd.DatetimeIndex(target_dates)):
        distance = np.abs(reference_days - date.dayofyear)
        distance = np.minimum(distance, 366 - distance)
        nearby = reference_ntl[distance <= CALENDAR_WINDOW_DAYS]

        matches[i] = len(nearby)
        if len(nearby) >= MIN_CALENDAR_MATCHES:
            predictions[i] = np.median(nearby)

    return predictions, matches

cutoff = EVENT_DATE - pd.Timedelta(days=BAU_HOLDOUT_DAYS)
future_dates = EVENT_DATE + pd.to_timedelta(
    future_blocks * AGGREGATION_DAYS + 1.5, unit="D"
)

bau_rows = []
audit_rows = []

for item in profile_meta.itertuples():
    baseline = four_day.loc[
        four_day.profile_id.eq(item.profile_id)
        & four_day.date_end.le(PRE_EVENT_END)
        & four_day.ntl_rq.notna()
    ].sort_values("date")

    if len(baseline) < MIN_STL_OBSERVED_BLOCKS:
        continue

    train = baseline[baseline.date_start.lt(cutoff)]
    holdout = baseline[baseline.date_start.ge(cutoff)]

    flat_mae = np.nan
    calendar_mae = np.nan
    method = "flat median"

    if len(train) >= MIN_STL_OBSERVED_BLOCKS and len(holdout) >= 8:
        train_recent = train[
            train.date_start.ge(cutoff - pd.Timedelta(days=365))
        ]
        if len(train_recent) < MIN_STL_OBSERVED_BLOCKS:
            train_recent = train

        flat_train = float(train_recent.ntl_rq.median())
        calendar_test, _ = calendar_median(
            holdout.date, train, flat_train
        )

        actual_test = holdout.ntl_rq.to_numpy(dtype=float)
        flat_mae = float(np.mean(np.abs(actual_test - flat_train)))
        calendar_mae = float(np.mean(np.abs(actual_test - calendar_test)))

        # Require a meaningful improvement before using seasonality.
        if calendar_mae < 0.90 * flat_mae:
            method = "calendar median"

        chosen_test = (
            calendar_test if method == "calendar median"
            else np.full(len(holdout), flat_train)
        )
        errors = actual_test - chosen_test
    else:
        errors = None

    recent = baseline[
        baseline.date_start.ge(EVENT_DATE - pd.Timedelta(days=365))
    ]
    if len(recent) < MIN_STL_OBSERVED_BLOCKS:
        recent = baseline

    flat_level = float(recent.ntl_rq.median())
    calendar_values, matches = calendar_median(
        future_dates, baseline, flat_level
    )

    expected = (
        calendar_values if method == "calendar median"
        else np.full(len(future_blocks), flat_level)
    )

    # Descriptive range from held-out errors; not a confidence interval.
    if errors is None:
        errors = baseline.ntl_rq.to_numpy(dtype=float) - flat_level
    error_lo, error_hi = np.quantile(errors, [.10, .90])

    for block, date, value, n_matches in zip(
        future_blocks, future_dates, expected, matches
    ):
        bau_rows.append(dict(
            profile_id=item.profile_id,
            block=int(block),
            date=date,
            expected_ntl=float(value),
            flat_trend_ntl=flat_level,  # Existing downstream column name
            variability_low=max(0., float(value + error_lo)),
            variability_high=max(0., float(value + error_hi)),
            bau_method=method,
            calendar_matches=int(n_matches),
        ))

    audit_rows.append(dict(
        profile_id=item.profile_id,
        label=item.label,
        scale=item.scale,
        bau_method=method,
        baseline_observed_blocks=len(baseline),
        holdout_observed_blocks=len(holdout),
        flat_holdout_mae=flat_mae,
        calendar_holdout_mae=calendar_mae,
    ))

calendar_forecast = pd.DataFrame(bau_rows)
bau_audit = pd.DataFrame(audit_rows)

calendar_forecast.to_csv(
    TABLE_DIR / "calendar_median_bau_scenarios.csv", index=False
)
bau_audit.to_csv(
    TABLE_DIR / "bau_holdout_selection.csv", index=False
)

display(bau_audit.sort_values(["scale", "label"]).round(2))

The existing refined panel design is defined once below. Its traces, map styling, legends, and annotation settings are reused unchanged for every spatial group.

In [ ]:
# PREVIEW_ID = "municipality::Tacloban"  # Set to None to export all profiles.
PREVIEW_ID = None  # Set to None to export all profiles.


show_profiles = {
    "regional::G3",
    "municipality::Tacloban",
    "centre5::Tacloban",
    "ghsl::30",
    "pixel1::Airport",
}

plot_start = dates.min()
plot_end = min(
    dates.max(),
    EVENT_DATE + pd.Timedelta(days=FORECAST_DAYS),
)

# Pre-Haiyan reference NTL on the original pixel grid.
reference_grid = np.full(land.shape, np.nan, dtype=float)
reference_grid.ravel()[
    land_pixels[reference_ok]
] = ntl0[reference_ok]

# GHSL display includes water.
ghsl_codes = [10, 11, 12, 13, 21, 22, 23, 30]
ghsl_colors = [
    "#BBD7EA",
    *[GHSL_CLASS_COLORS[code] for code in ghsl_codes[1:]],
]
ghsl_grid = np.full(land.shape, np.nan)

for index, code in enumerate(ghsl_codes):
    ghsl_grid[ghsl == code] = index

ghsl_scale = [
    [position, color]
    for index, color in enumerate(ghsl_colors)
    for position in (
        index / len(ghsl_codes),
        (index + 1) / len(ghsl_codes),
    )
]

dx = float(abs(a2.x.values[1] - a2.x.values[0]))
dy = float(abs(a2.y.values[1] - a2.y.values[0]))

# Light municipal borders.
boundary_x, boundary_y = [], []
for geometry in region.geometry:
    edge = geometry.boundary
    segments = edge.geoms if hasattr(edge, "geoms") else [edge]

    for segment in segments:
        xx, yy = segment.xy
        boundary_x.extend([*xx, None])
        boundary_y.extend([*yy, None])

# Haiyan path from the shapefile used in the earlier notebooks.
track_path = (
    DATA_DIR
    / "yolanda-path-line-"
    / "Yolanda Path Line.shp"
)
track_x, track_y = [], []

if track_path.exists():
    haiyan_track = gpd.read_file(
        track_path
    ).to_crs(a2.rio.crs)

    for geometry in haiyan_track.geometry.dropna():
        segments = (
            geometry.geoms
            if hasattr(geometry, "geoms")
            else [geometry]
        )

        for segment in segments:
            if hasattr(segment, "xy"):
                xx, yy = segment.xy
                track_x.extend([*xx, None])
                track_y.extend([*yy, None])
else:
    print("Haiyan path shapefile not found; path omitted.")

sc_colorscale = [
    [0.00, "#F3F5F8"],
    [0.10, "#DCEEDD"],
    [0.50, "#75BD83"],
    [1.00, "#006B2D"],
]


def select_plot_meta(scale):
    group = profile_meta.loc[profile_meta.scale.eq(scale)]
    if PREVIEW_ID is not None:
        group = group.loc[group.profile_id.eq(PREVIEW_ID)]
    return group

def render_profiles(plot_meta):
    if plot_meta.empty:
        return
    saved_count = 0

    for item in tqdm(
        plot_meta.itertuples(),
        total=len(plot_meta),
        desc="STL figures",
    ):
        pid = item.profile_id

        fit = stl_components[
            stl_components.profile_id.eq(pid)
        ].sort_values("date")
        if fit.empty:
            continue

        obs = four_day[
            four_day.profile_id.eq(pid)
        ].sort_values("date")
        obs_short = obs[obs.date.le(plot_end)].copy()

        future_dates = pd.DatetimeIndex(
            forecast.loc[
                forecast.profile_id.eq(pid)
                & forecast.date.le(plot_end),
                "date",
            ].sort_values()
        )
        if len(future_dates) == 0:
            continue

        display_color = {
            "#ffff00": "#B49B00",
            "#cdf57a": "#789A36",
            "#abcd66": "#789A36",
        }.get(str(item.color).lower(), item.color)

        # ------------------------------------------------------------
        # Curved scenario from the pre-event STL fit
        # ------------------------------------------------------------

        pre_dates = pd.DatetimeIndex(fit.date)
        fit_log = (
            fit.trend_log.to_numpy(dtype=float)
            + fit.seasonal_log.to_numpy(dtype=float)
        )

        fit_envelope = (
            pd.Series(fit_log)
            .rolling(
                window=STL_PERIOD_BLOCKS,
                center=True,
                min_periods=1,
            )
            .mean()
            .to_numpy()
        )

        analogue_dates = (
            future_dates - pd.Timedelta(days=365)
        )
        pre_day = np.asarray(
            (pre_dates - pre_dates[0])
            / pd.Timedelta(days=1),
            dtype=float,
        )
        analogue_day = np.asarray(
            (analogue_dates - pre_dates[0])
            / pd.Timedelta(days=1),
            dtype=float,
        )

        prior_year_curve = np.interp(
            analogue_day,
            pre_day,
            fit_envelope,
        )
        expected_log = (
            fit_envelope[-1]
            + prior_year_curve
            - prior_year_curve[0]
        )
        expected_ntl = np.maximum(
            0,
            np.expm1(expected_log),
        )

        baseline_departure = (
            np.log1p(
                fit.observed_ntl.to_numpy(dtype=float)
            )
            - fit_envelope
        )
        valid_departure = baseline_departure[
            np.isfinite(baseline_departure)
        ]
        departure_low, departure_high = np.quantile(
            valid_departure,
            [.10, .90],
        )

        range_low = np.maximum(
            0,
            np.expm1(expected_log + departure_low),
        )
        range_high = np.maximum(
            0,
            np.expm1(expected_log + departure_high),
        )

        # Departure is RQ NTL minus the curved projected STL fit.
        # It does not use the recent-median line.
        projected = pd.DataFrame({
            "date": future_dates,
            "expected_ntl": expected_ntl,
        })
        departure = (
            obs_short.loc[
                obs_short.date.ge(EVENT_DATE),
                ["date", "ntl_rq"],
            ]
            .merge(projected, on="date", how="left")
        )
        departure["difference_ntl"] = (
            departure.ntl_rq
            - departure.expected_ntl
        )

        recent = obs.loc[
            obs.date.ge(
                EVENT_DATE - pd.Timedelta(days=365)
            )
            & obs.date.lt(EVENT_DATE),
            "ntl_rq",
        ].dropna()

        # ------------------------------------------------------------
        # Left: SC, NTL, STL + departure.
        # Right: matched GHSL and NTL maps.
        # ------------------------------------------------------------

        fig = make_subplots(
            rows=6,
            cols=2,
            specs=[
                [{}, {"rowspan": 3}],
                [{"rowspan": 3}, None],
                [None, None],
                [None, {"rowspan": 3}],
                [{"rowspan": 2, "secondary_y": True}, None],
                [None, None],
            ],
            column_widths=[.70, .30],
            row_heights=[
                .10, .19, .19,
                .19, .165, .165,
            ],
            horizontal_spacing=.07,
            vertical_spacing=.028,
        )

        left_domain = fig.layout.xaxis.domain
        sc_domain = fig.layout.yaxis.domain
        ntl_domain = fig.layout.yaxis3.domain
        components_domain = fig.layout.yaxis5.domain
        ghsl_map_domain = fig.layout.yaxis2.domain
        ntl_map_domain = fig.layout.yaxis4.domain

        # ------------------------------------------------------------
        # Left, top: shaded spatial completeness
        # ------------------------------------------------------------

        fig.add_trace(go.Heatmap(
            x=obs_short.date,
            y=["SC"],
            z=obs_short.sc_pct.to_numpy(
                dtype=float
            )[None, :],
            colorscale=sc_colorscale,
            zmin=0,
            zmax=100,
            hoverongaps=False,
            showscale=True,
            colorbar=dict(
                title=dict(
                    text="SC (%)",
                    font=dict(size=14),
                ),
                x=left_domain[1] + .003,
                xanchor="left",
                y=sum(sc_domain) / 2,
                len=sc_domain[1] - sc_domain[0],
                thickness=15,
                tickvals=[0, 50, 100],
                tickfont=dict(size=13),
            ),
            hovertemplate=(
                "%{x|%d %b %Y}"
                "<br>SC %{z:.0f}%"
                "<extra></extra>"
            ),
        ), row=1, col=1)

        # ------------------------------------------------------------
        # Left, middle: observed NTL and curved scenario
        # ------------------------------------------------------------

        fig.add_trace(go.Scatter(
            x=future_dates,
            y=range_low,
            mode="lines",
            line=dict(width=0),
            hoverinfo="skip",
            showlegend=False,
        ), row=2, col=1)

        fig.add_trace(go.Scatter(
            x=future_dates,
            y=range_high,
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(213,94,0,.12)",
            name="Pre-event spread",
            legend="legend2",
            legendrank=5,
            hoverinfo="skip",
        ), row=2, col=1)

        fig.add_trace(go.Scatter(
            x=fit.date,
            y=fit.fitted_ntl,
            mode="lines",
            line=dict(
                color="#243B5A",
                width=2,
            ),
            name="STL fit",
            legend="legend2",
            legendrank=2,
        ), row=2, col=1)

        fig.add_trace(go.Scatter(
            x=future_dates,
            y=expected_ntl,
            mode="lines",
            line=dict(
                color="#D55E00",
                width=2.7,
            ),
            name="Projected STL fit",
            legend="legend2",
            legendrank=3,
        ), row=2, col=1)

        if not recent.empty:
            fig.add_trace(go.Scatter(
                x=future_dates,
                y=np.full(
                    len(future_dates),
                    float(recent.median()),
                ),
                mode="lines",
                line=dict(
                    color="#81909F",
                    width=1.5,
                    dash="dot",
                ),
                name="Recent median",
                legend="legend2",
                legendrank=4,
            ), row=2, col=1)

        fig.add_trace(go.Scatter(
            x=obs_short.date,
            y=obs_short.ntl_rq,
            mode="lines+markers",
            connectgaps=False,
            line=dict(
                color=display_color,
                width=1.8,
            ),
            marker=dict(
                color=display_color,
                size=3.5,
            ),
            name="RQ NTL (4D)",
            legend="legend2",
            legendrank=1,
            customdata=obs_short[
                ["sc_pct", "n_valid", "n_fixed"]
            ],
            hovertemplate=(
                "%{x|%d %b %Y}"
                "<br>RQ NTL %{y:.2f}"
                "<br>SC %{customdata[0]:.0f}%"
                "<br>Pixels %{customdata[1]:.0f}/"
                "%{customdata[2]:.0f}"
                "<extra></extra>"
            ),
        ), row=2, col=1)

        # ------------------------------------------------------------
        # Left, bottom: STL components and departure
        # ------------------------------------------------------------

        fig.add_trace(go.Scatter(
            x=fit.date,
            y=fit.trend_log,
            mode="lines",
            line=dict(
                color="#243B5A",
                width=2.2,
            ),
            name="Trend",
            legend="legend3",
            legendrank=1,
        ), row=5, col=1, secondary_y=False)

        fig.add_trace(go.Scatter(
            x=fit.date,
            y=fit.seasonal_log,
            mode="lines",
            line=dict(
                color="#009E73",
                width=1.7,
            ),
            name="28D component",
            legend="legend3",
            legendrank=2,
        ), row=5, col=1, secondary_y=False)

        fig.add_trace(go.Scatter(
            x=fit.date,
            y=fit.residual_log,
            mode="markers",
            marker=dict(
                color="#CC79A7",
                size=4,
                opacity=.75,
            ),
            name="Residual",
            legend="legend3",
            legendrank=3,
        ), row=5, col=1, secondary_y=False)

        fig.add_trace(go.Scatter(
            x=departure.date,
            y=departure.difference_ntl,
            mode="lines+markers",
            connectgaps=False,
            line=dict(
                color="#0072B2",
                width=1.9,
            ),
            marker=dict(
                color="#0072B2",
                size=3.5,
            ),
            name="RQ NTL − projected fit",
            legend="legend3",
            legendrank=4,
            hovertemplate=(
                "%{x|%d %b %Y}"
                "<br>RQ NTL − projected fit "
                "%{y:+.2f}"
                "<extra></extra>"
            ),
        ), row=5, col=1, secondary_y=True)

        fig.add_trace(go.Scatter(
            x=[plot_start, plot_end],
            y=[0, 0],
            mode="lines",
            line=dict(
                color=BASELINE_COLOR,
                width=1,
            ),
            hoverinfo="skip",
            showlegend=False,
        ), row=5, col=1, secondary_y=True)

        # ------------------------------------------------------------
        # One geographic extent for both maps
        # ------------------------------------------------------------

        if item.scale in ("ghsl", "regional"):
            x0, y0, x1, y1 = map(
                float,
                region.total_bounds,
            )

        elif item.scale == "municipality":
            selected = focused[
                focused.unit_name.eq(item.municipality)
            ]
            geometry = selected.geometry.iloc[0]
            x0, y0, x1, y1 = geometry.bounds

        else:
            lon = float(item.longitude)
            lat = float(item.latitude)
            radius = (
                .065 if item.scale == "pixel1"
                else .085
            )
            x0 = lon - radius
            x1 = lon + radius
            y0 = lat - radius
            y1 = lat + radius

        cx = (x0 + x1) / 2
        cy = (y0 + y1) / 2
        cos_lat = np.cos(np.deg2rad(cy))

        side_lat = 1.12 * max(
            (x1 - x0) * cos_lat,
            y1 - y0,
        )
        if item.scale == "pixel1":
            side_lat = max(side_lat, .14)
        elif item.scale == "kernel5":
            side_lat = max(side_lat, .18)

        bounds = (
            cx - side_lat / (2 * cos_lat),
            cy - side_lat / 2,
            cx + side_lat / (2 * cos_lat),
            cy + side_lat / 2,
        )

        map_step = 3 if item.scale in ("ghsl", "regional") else 1
        xi = np.flatnonzero(
            (a2.x.values >= bounds[0])
            & (a2.x.values <= bounds[2])
        )[::map_step]
        yi = np.flatnonzero(
            (a2.y.values >= bounds[1])
            & (a2.y.values <= bounds[3])
        )[::map_step]

        # Upper-right GHSL map.
        fig.add_trace(go.Heatmap(
            x=a2.x.values[xi],
            y=a2.y.values[yi],
            z=ghsl_grid[np.ix_(yi, xi)],
            colorscale=ghsl_scale,
            zmin=-.5,
            zmax=len(ghsl_codes) - .5,
            zsmooth=False,
            showscale=True,
            colorbar=dict(
                title=dict(
                    text="GHSL",
                    font=dict(size=14),
                ),
                y=sum(ghsl_map_domain) / 2,
                len=(ghsl_map_domain[1]
                     - ghsl_map_domain[0]) * .72,
                thickness=15,
                tickvals=list(range(len(ghsl_codes))),
                ticktext=[str(code) for code in ghsl_codes],
                tickfont=dict(size=13),
            ),
            customdata=ghsl[np.ix_(yi, xi)],
            hoverongaps=False,
            hovertemplate=(
                "%{x:.3f}°E, %{y:.3f}°N"
                "<br>GHSL %{customdata:.0f}"
                "<extra></extra>"
            ),
        ), row=1, col=2)

        # Lower-right reference NTL map.
        fig.add_trace(go.Heatmap(
            x=a2.x.values[xi],
            y=a2.y.values[yi],
            z=reference_grid[np.ix_(yi, xi)],
            colorscale="Inferno",
            zmin=0,
            zmax=5,
            zsmooth=False,
            showscale=True,
            colorbar=dict(
                title=dict(
                    text="NTL<br>nW cm⁻² sr⁻¹",
                    font=dict(size=14),
                ),
                y=sum(ntl_map_domain) / 2,
                len=(ntl_map_domain[1]
                     - ntl_map_domain[0]) * .72,
                thickness=15,
                tickfont=dict(size=13),
            ),
            hoverongaps=False,
            hovertemplate=(
                "%{x:.3f}°E, %{y:.3f}°N"
                "<br>Reference NTL %{z:.2f}"
                "<extra></extra>"
            ),
        ), row=4, col=2)

        # Context and selected support on both maps.
        for map_row, axis_number in (
            (1, "2"),
            (4, "4"),
        ):
            fig.add_trace(go.Scatter(
                x=boundary_x,
                y=boundary_y,
                mode="lines",
                line=dict(
                    color=(
                        "rgba(36,59,90,.45)"
                        if map_row == 1
                        else "rgba(185,200,218,.65)"
                    ),
                    width=.7,
                ),
                hoverinfo="skip",
                showlegend=False,
            ), row=map_row, col=2)

            if item.scale in ("pixel1", "kernel5"):
                support = fixed_profiles[pid]
                iy_support, ix_support = np.unravel_index(
                    land_pixels[support],
                    land.shape,
                )
                selected_x = a2.x.values[ix_support]
                selected_y = a2.y.values[iy_support]

                box = (
                    float(selected_x.min() - dx / 2),
                    float(selected_y.min() - dy / 2),
                    float(selected_x.max() + dx / 2),
                    float(selected_y.max() + dy / 2),
                )

                for color, width in (
                    ("white", 5),
                    ("#111827", 2.5),
                ):
                    fig.add_shape(
                        type="rect",
                        xref=f"x{axis_number}",
                        yref=f"y{axis_number}",
                        x0=box[0],
                        y0=box[1],
                        x1=box[2],
                        y1=box[3],
                        line=dict(
                            color=color,
                            width=width,
                        ),
                        fillcolor="rgba(0,0,0,0)",
                    )

            elif item.scale == "municipality":
                edge = geometry.boundary
                segments = (
                    edge.geoms
                    if hasattr(edge, "geoms")
                    else [edge]
                )
                for segment in segments:
                    xx, yy = segment.xy

                    for color, width in (
                        ("white", 4),
                        ("#111827", 2),
                    ):
                        fig.add_trace(go.Scatter(
                            x=list(xx),
                            y=list(yy),
                            mode="lines",
                            line=dict(
                                color=color,
                                width=width,
                            ),
                            hoverinfo="skip",
                            showlegend=False,
                        ), row=map_row, col=2)

            elif item.scale == "ghsl":
                class_mask = (
                    (ghsl[np.ix_(yi, xi)]
                     == int(item.ghsl_code))
                    & (regional_zone[np.ix_(yi, xi)] > 0)
                ).astype(float)

                fig.add_trace(go.Contour(
                    x=a2.x.values[xi],
                    y=a2.y.values[yi],
                    z=class_mask,
                    contours=dict(
                        start=.5,
                        end=.5,
                        size=1,
                        coloring="lines",
                        showlabels=False,
                    ),
                    line=dict(
                        color="white",
                        width=1.2,
                    ),
                    showscale=False,
                    hoverinfo="skip",
                    showlegend=False,
                ), row=map_row, col=2)

            if track_x:
                for color, width in (
                    ("white", 4),
                    ("#111827", 2),
                ):
                    fig.add_trace(go.Scatter(
                        x=track_x,
                        y=track_y,
                        mode="lines",
                        line=dict(
                            color=color,
                            width=width,
                            dash="dash",
                        ),
                        hovertemplate=(
                            "Haiyan path<extra></extra>"
                        ),
                        showlegend=False,
                    ), row=map_row, col=2)

        # ------------------------------------------------------------
        # Aligned time axes and undistorted maps
        # ------------------------------------------------------------

        for row in (1, 2, 5):
            fig.update_xaxes(
                range=[plot_start, plot_end],
                tickformat="%b %Y",
                dtick="M3",
                tickfont=dict(size=12),
                showticklabels=(row == 5),
                row=row,
                col=1,
            )

        fig.update_yaxes(
            showticklabels=False,
            showgrid=False,
            row=1,
            col=1,
        )
        fig.update_yaxes(
            title_text="NTL (nW cm⁻² sr⁻¹)",
            rangemode="tozero",
            tickfont=dict(size=12),
            row=2,
            col=1,
        )
        fig.update_yaxes(
            title_text="STL · log(1 + NTL)",
            tickfont=dict(size=12),
            row=5,
            col=1,
            secondary_y=False,
        )
        fig.update_yaxes(
            title_text="NTL difference",
            tickfont=dict(size=12),
            row=5,
            col=1,
            secondary_y=True,
        )

        for row in (1, 4):
            fig.update_xaxes(
                range=[bounds[0], bounds[2]],
                showgrid=False,
                zeroline=False,
                nticks=3,
                tickfont=dict(size=11),
                ticksuffix="°E",
                constrain="domain",
                row=row,
                col=2,
            )
            fig.update_yaxes(
                range=[bounds[1], bounds[3]],
                showgrid=False,
                zeroline=False,
                nticks=3,
                tickfont=dict(size=11),
                ticksuffix="°N",
                scaleanchor=(
                    "x2" if row == 1 else "x4"
                ),
                scaleratio=1 / cos_lat,
                constrain="domain",
                row=row,
                col=2,
            )

        for row in (1, 2, 5):
            fig.add_vline(
                x=EVENT_DATE,
                line_color=EVENT_COLOR,
                line_dash="dash",
                line_width=1.4,
                row=row,
                col=1,
            )

        style_figure(
            fig,
            width=1600,
            height=1000,
            bottom=90,
        )

        # Position colorbars after map aspect constraints are applied.
        for trace in fig.data:
            if trace.type != "heatmap":
                continue

            if trace.xaxis == "x":
                trace.colorbar.x = (
                    fig.layout.xaxis.domain[1] + .003
                )
            elif trace.xaxis == "x2":
                trace.colorbar.x = (
                    fig.layout.xaxis2.domain[1] + .005
                )
            elif trace.xaxis == "x4":
                trace.colorbar.x = (
                    fig.layout.xaxis4.domain[1] + .005
                )

            trace.colorbar.xanchor = "left"

        fig.update_layout(
            title=dict(
                text=item.label,
                x=.08,
                y=.92,
                font=dict(size=24),
            ),
            margin=dict(
                l=100,
                r=105,
                t=85,
                b=85,
            ),
            legend2=dict(
                orientation="h",
                x=left_domain[0],
                y=ntl_domain[1] - .008,
                xanchor="left",
                yanchor="bottom",
                font=dict(size=15),
                bgcolor="rgba(255,255,255,.94)",
            ),
            legend3=dict(
                orientation="h",
                x=left_domain[0],
                y=components_domain[1] - .008,
                xanchor="left",
                yanchor="bottom",
                font=dict(size=15),
                bgcolor="rgba(255,255,255,.94)",
            ),
        )

        for x_axis, y_axis, label in (
            (fig.layout.xaxis2, fig.layout.yaxis2, "GHSL settlement"),
            (fig.layout.xaxis4, fig.layout.yaxis4, "Pre-Haiyan reference NTL"),
        ):
            fig.add_annotation(
                x=sum(x_axis.domain) / 2,
                y=y_axis.domain[1] - .015,
                xref="paper",
                yref="paper",
                xanchor="center",
                yanchor="bottom",
                text=label,
                showarrow=False,
                font=dict(size=18, color=PLOT_TEXT_COLOR),
            )

        fig.layout.margin.t = 105

        filename = (
            "03_stl_"
            + pid.replace("::", "_")
                 .replace(" ", "_")
                 .replace("/", "_")
        )

        if PREVIEW_ID is None:
            finish_figure(
                fig,
                filename,
                show=pid in show_profiles,
            )
            saved_count += 1
        else:
            fig.show()

    if PREVIEW_ID is None:
        print(
            f"Saved {saved_count} profile-specific figures "
            f"in {FIGURE_DIR}."
        )

## 4. Regional G3 overview

The first full-scale figure shows spatial completeness, observed NTL, the projected STL-fit curve, decomposition, observed departure, and both context maps for the combined G3 support.

In [ ]:
render_profiles(select_plot_meta("regional"))

## 5. Municipalities

Municipal plots use G3 support within each boundary. Compare the STL scenario with observed-pair recurrence and the municipality-specific baseline checks before reading departures.

Observed-pair correlations check short and approximately annual lags without filling missing blocks.

In [ ]:
# Municipality-scale observed-lag correlations
municipal = profile_meta.loc[
    profile_meta.scale.eq("municipality"),
    ["profile_id", "label", "color"]
].copy()
municipal_ids = municipal.profile_id.tolist()

lag_grid = (
    lag_table[lag_table.profile_id.isin(municipal_ids)]
    .pivot(index="profile_id", columns="lag_days", values="correlation")
    .reindex(municipal_ids)
)
pair_grid = (
    lag_table[lag_table.profile_id.isin(municipal_ids)]
    .pivot(index="profile_id", columns="lag_days", values="paired_blocks")
    .reindex(municipal_ids)
)

values = lag_grid.to_numpy(dtype=float)
finite = values[np.isfinite(values)]
color_limit = max(.3, np.abs(finite).max()) if finite.size else .3

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[.62, .38],
    horizontal_spacing=.12,
    subplot_titles=["20–80-day lags", "Near-annual lags · exploratory"],
)

for col, selected in (
    (1, lag_grid.columns[lag_grid.columns <= 80]),
    (2, lag_grid.columns[lag_grid.columns >= 336]),
):
    fig.add_trace(
        go.Heatmap(
            x=selected,
            y=municipal.label,
            z=lag_grid[selected].values,
            customdata=pair_grid[selected].values,
            colorscale="RdBu",
            reversescale=True,
            zmin=-color_limit,
            zmax=color_limit,
            showscale=(col == 2),
            colorbar=dict(
                title=dict(text="Correlation", font=dict(size=16)),
                tickfont=dict(size=14),
                thickness=18,
                len=.8,
                y=.5,
            ),
            hovertemplate=(
                "%{y}<br>Lag %{x} days"
                "<br>Correlation %{z:.2f}"
                "<br>Observed pairs %{customdata:.0f}<extra></extra>"
            ),
        ),
        row=1, col=col,
    )
    fig.update_xaxes(
        title_text="Lag (days)",
        tickfont=dict(size=14),
        row=1, col=col,
    )
    fig.update_yaxes(
        autorange="reversed",
        tickfont=dict(size=14),
        showticklabels=(col == 1),
        row=1, col=col,
    )

style_figure(fig, width=1500, height=720, bottom=95)
fig.update_layout(margin=dict(l=175, r=120, t=65, b=95))
finish_figure(fig, "04_municipal_observed_lag_correlations")

Seasonal strength from STL is compared with recurrence calculated only from observed pairs; agreement is evidence to examine, not proof of a physical cycle.

In [ ]:
# Municipality-scale STL strength versus recurrence in observed data
best_short = (
    lag_table[
        lag_table.profile_id.isin(municipal_ids)
        & lag_table.lag_days.between(20, 80)
    ]
    .dropna(subset=["correlation"])
    .sort_values(["profile_id", "correlation"], ascending=[True, False])
    .drop_duplicates("profile_id")
    [["profile_id", "lag_days"]]
    .rename(columns={"lag_days": "strongest_lag_days"})
)

seasonality = (
    model_audit[model_audit.scale.eq("municipality")]
    .merge(municipal[["profile_id", "color"]], on="profile_id")
    .merge(best_short, on="profile_id", how="left")
    .set_index("profile_id")
    .reindex(municipal_ids)
    .reset_index()
)
seasonality["baseline_observed_share_pct"] = (
    100 * seasonality.baseline_observed_blocks
    / seasonality.baseline_calendar_blocks
)

fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=.12,
    subplot_titles=[
        "STL 28-day strength",
        "Strongest recurrence in observed blocks",
    ],
)

fig.add_trace(
    go.Scatter(
        x=seasonality.seasonal_strength,
        y=seasonality.label,
        mode="markers",
        marker=dict(
            size=13, color=seasonality.color,
            line=dict(color="white", width=1),
        ),
        customdata=seasonality[
            ["baseline_observed_blocks", "baseline_calendar_blocks",
             "baseline_interpolated_pct"]
        ],
        hovertemplate=(
            "%{y}<br>STL strength %{x:.2f}"
            "<br>Observed baseline %{customdata[0]:.0f}/%{customdata[1]:.0f} blocks"
            "<br>Interpolated %{customdata[2]:.1f}%<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(
        x=seasonality.peak_lag_correlation,
        y=seasonality.label,
        mode="markers",
        marker=dict(
            size=13, color=seasonality.color,
            line=dict(color="white", width=1),
        ),
        customdata=seasonality[["strongest_lag_days", "peak_lag_pairs"]],
        hovertemplate=(
            "%{y}<br>Strongest correlation %{x:.2f}"
            "<br>Lag %{customdata[0]:.0f} days"
            "<br>Observed pairs %{customdata[1]:.0f}<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1, col=2,
)

fig.add_vline(
    x=.2, line_color=BASELINE_COLOR, line_dash="dot",
    line_width=1.3, row=1, col=2,
)
fig.update_xaxes(
    title_text="STL seasonal strength",
    range=[0, 1], tickfont=dict(size=14), row=1, col=1,
)
fig.update_xaxes(
    title_text="Observed-pair correlation",
    range=[-1, 1], tickfont=dict(size=14), row=1, col=2,
)
for col in (1, 2):
    fig.update_yaxes(
        categoryorder="array",
        categoryarray=municipal.label.tolist(),
        autorange="reversed",
        tickfont=dict(size=14),
        showticklabels=(col == 1),
        row=1, col=col,
    )

style_figure(fig, width=1450, height=720, bottom=95)
fig.update_layout(margin=dict(l=175, r=55, t=65, b=95))
finish_figure(fig, "05_municipal_seasonality_evidence")

The evidence table keeps observed-block counts beside the fitted seasonal measures.

In [ ]:
# Municipality-scale evidence table
municipal_table = seasonality[[
    "label",
    "fit_status",
    "baseline_observed_blocks",
    "baseline_calendar_blocks",
    "baseline_observed_share_pct",
    "baseline_interpolated_pct",
    "seasonal_strength",
    "strongest_lag_days",
    "peak_lag_correlation",
    "peak_lag_pairs",
    "annual_lag_peak_days",
    "annual_lag_correlation",
    "annual_lag_pairs",
]].copy()

municipal_table.to_csv(
    TABLE_DIR / "municipal_baseline_seasonality.csv",
    index=False,
)
display(municipal_table.round(2))

### Why validate a municipal baseline

The useful baseline is the one that predicts withheld, observable pre-Haiyan NTL—not the one that produces the smoothest fit. A curved STL trend describes the past. Its future curvature is unknown unless a repeatable mechanism or an unaffected comparison region supplies evidence for it. Your municipality analysis should therefore compare a small set of business-as-usual (BAU) models using pseudo-events before Haiyan, then measure post-event departures only where reliability-qualified NTL was observed.

This is a meaningful extension of RQ1: observability determines which values can test a baseline; forecasting performance determines whether that baseline can support an impact claim. Your RQ1 work shows that gaps and spatial completeness are structured, especially in cloud-affected tropical regions. Treating missing blocks as zero would manufacture both model skill and disaster losses (Principe et al., 2026).

| Established approach | What it contributes | Decision for these data |
|---|---|---|
| **Recent pre-event reference** | A transparent level against which change can be measured. Black Marble disaster studies have used pre-storm radiance references and four-day aggregation; a Philippine Haiyan analysis used a six-month pre-event mean. ([Román et al., 2019](https://pmc.ncbi.nlm.nih.gov/articles/PMC6599127/); [Philippine Space Agency](https://philsa.gov.ph/research/spatiotemporal-tip-and-cue-method-for-proactive-post-disaster-assessment-using-nighttime-lights/)) | Use the median of observed four-day blocks in the preceding 180 days as the **benchmark**. It assumes no future trend. |
| **Local-level state-space model** | Estimates a changing underlying level while leaving unavailable blocks missing. State-space forecasting can work with missing observations. ([statsmodels structural models](https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.structural.UnobservedComponents.html); [missing-data example](https://www.statsmodels.org/stable/examples/notebooks/generated/statespace_sarimax_internet.html)) | Test as the main flexible alternative. A level that curved historically may still have a flat *expected* future; uncertainty grows rather than a particular curve being assumed. |
| **28-day state-space or STL forecast** | Tests whether the apparent short cycle improves prediction. STL needs a **separate forecasting rule** for its components. ([Forecasting: Principles and Practice](https://otexts.com/fpp3/forecasting-decomposition.html); [statsmodels STL forecasting](https://www.statsmodels.org/stable/generated/statsmodels.tsa.forecasting.stl.STLForecast.html)) | Keep both as challengers. A seven-block period is imposed by the model; predictive success does not, by itself, prove a physical 28-day lighting cycle. STL interpolation remains confined to model fitting. |
| **Synthetic control** | Can estimate BAU from unaffected places that matched the municipality before the event; this has been applied to daily Black Marble disaster assessment. ([Mu et al., 2025](https://ira.lib.polyu.edu.hk/handle/10397/112798)) | **Not yet defensible with the current municipal set:** the Samar–Leyte municipalities cannot simply be presumed untreated donors. Screen external regions and pre-event matching before adding this method. |

Keep the existing four-day reliability-qualified DNB_BRDF_Corrected_NTL, fixed reference pixels, GHSL 22/23/30 municipal support, and SC ≥10%. Do not switch to monthly composites to make a forecast easier. Longer composites improve valid coverage but can materially alter radiance magnitude and timing; a 4-day model should be validated at the 4-day scale at which disruption will be measured (Zheng et al., 2022).
The code below uses three pre-Haiyan pseudo-events, each followed by a 180-day withheld period. It reports errors at 30, 90, and 180 days, chooses a municipality’s model using 90-day observed-block MAE, and prefers the simpler model when its error is within 5% of the best. Those choices are explicit research rules, not literature constants. Rolling-origin, horizon-matched validation is preferable to judging in-sample fit, and MAE remains meaningful when radiance approaches zero (forecast cross-validation; accuracy measures).
Two limits matter when interpreting the results:
- The current fixed municipal pixels were selected using the 60 days before Haiyan. Pseudo-event tests therefore validate temporal prediction conditional on that final spatial support; they are not a fully prospective test of pixel selection.
- A model trained on January 2012–November 2013 cannot establish a credible no-Haiyan trajectory through 2024 on its own. The first 180 days are the immediate application here. Longer recovery claims need additional controls or an explicitly updated reference, with uncertainty stated separately.
Run these cells after four_day, profile_meta, lag_table, STL, and the existing plotting helpers have been created. They do not rerun pixel extraction or alter the notebook’s existing forecasts.

Four municipal BAU candidates are fitted on the same qualified four-day series; missing observations remain missing.

In [ ]:
# Municipal BAU candidates: observed NTL stays NaN when unavailable.
from statsmodels.tsa.statespace.structural import UnobservedComponents

MUNICIPAL_MODELS = (
    "Recent median",
    "Local level",
    "28-day state space",
    "28-day STL",
)
BACKTEST_ORIGINS = pd.to_datetime([
    "2012-11-01", "2013-02-01", "2013-05-01",
])
BACKTEST_DAYS = 180

municipal_meta = profile_meta.loc[
    profile_meta.scale.eq("municipality"),
    ["profile_id", "label", "color", "fixed_pixels"]
].copy()

municipal_series = (
    four_day[four_day.profile_id.isin(municipal_meta.profile_id)]
    .sort_values(["profile_id", "block"])
    .copy()
)
municipal_series["ntl_rq"] = municipal_series.ntl_rq.where(
    municipal_series.observed
)

def municipal_bau(train, steps, method):
    y = train.ntl_rq.to_numpy(dtype=float)
    if np.isfinite(y).sum() < 24:
        raise ValueError("Fewer than 24 observed training blocks")

    if method == "Recent median":
        recent = y[-45:]  # 180 calendar days
        if np.isfinite(recent).sum() < 10:
            raise ValueError("Fewer than 10 observed blocks in recent reference")
        return np.full(steps, np.nanmedian(recent))

    if method in ("Local level", "28-day state space"):
        specification = dict(level="local level")
        if method == "28-day state space":
            specification.update(seasonal=7, stochastic_seasonal=False)

        result = UnobservedComponents(
            np.log1p(y), **specification
        ).fit(disp=False, maxiter=100)

        if not result.mle_retvals.get("converged", True):
            raise RuntimeError("State-space fit did not converge")
        return np.maximum(0, np.expm1(np.asarray(result.forecast(steps))))

    if method == "28-day STL":
        filled = pd.Series(y).interpolate(
            limit_direction="both"
        ).to_numpy()
        result = STL(np.log1p(filled), period=7, robust=True).fit()
        blocks = train.block.to_numpy(dtype=int)
        seasonal = np.asarray(result.seasonal)

        phase_template = np.array([
            np.median(seasonal[blocks % 7 == phase])
            for phase in range(7)
        ])
        future_blocks = blocks[-1] + np.arange(1, steps + 1)
        recent_trend = np.median(np.asarray(result.trend)[-7:])

        return np.maximum(
            0,
            np.expm1(recent_trend + phase_template[future_blocks % 7]),
        )

    raise ValueError(f"Unknown model: {method}")

Pseudo-event tests choose a municipality-specific baseline using held-out observed blocks.

In [ ]:
# Pseudo-event validation and municipality-specific model choice.
backtest_rows, fit_failures = [], []

for item in municipal_meta.itertuples(index=False):
    g = municipal_series[
        municipal_series.profile_id.eq(item.profile_id)
        & municipal_series.date.lt(EVENT_DATE)
    ]

    for origin in BACKTEST_ORIGINS:
        train = g[g.date.lt(origin)]
        holdout = g[
            g.date.ge(origin)
            & g.date.lt(origin + pd.Timedelta(days=BACKTEST_DAYS))
        ]

        if train.ntl_rq.notna().sum() < 24 or holdout.ntl_rq.notna().sum() < 8:
            continue

        for method in MUNICIPAL_MODELS:
            try:
                predicted = municipal_bau(train, len(holdout), method)
            except (ValueError, RuntimeError, np.linalg.LinAlgError) as exc:
                fit_failures.append(dict(
                    profile_id=item.profile_id,
                    origin=origin,
                    model=method,
                    reason=str(exc),
                ))
                continue

            for position, block in enumerate(holdout.itertuples(index=False)):
                if not np.isfinite(block.ntl_rq):
                    continue
                backtest_rows.append(dict(
                    profile_id=item.profile_id,
                    origin=origin,
                    model=method,
                    date=block.date,
                    lead_days=(block.date - origin).days + 1,
                    observed_ntl=block.ntl_rq,
                    expected_ntl=predicted[position],
                    sc_pct=block.sc_pct,
                ))

backtest = pd.DataFrame(backtest_rows)
backtest["error"] = backtest.observed_ntl - backtest.expected_ntl
backtest["absolute_error"] = backtest.error.abs()

score_parts = []
for days in (30, 90, 180):
    part = (
        backtest[backtest.lead_days.le(days)]
        .groupby(["profile_id", "model"], as_index=False)
        .agg(
            mae=("absolute_error", "mean"),
            folds=("origin", "nunique"),
            scored_blocks=("absolute_error", "size"),
        )
    )
    part["horizon_days"] = days
    score_parts.append(part)

model_scores = pd.concat(score_parts, ignore_index=True)
scores_90 = model_scores[model_scores.horizon_days.eq(90)]

selection_rows = []
for item in municipal_meta.itertuples(index=False):
    available = scores_90[
        scores_90.profile_id.eq(item.profile_id)
        & scores_90.folds.ge(2)
        & scores_90.scored_blocks.ge(12)
    ]
    if available.empty:
        chosen = "Recent median"
        status = "insufficient pseudo-event evidence"
    else:
        near_best = available[
            available.mae.le(1.05 * available.mae.min())
        ]
        chosen = next(
            name for name in MUNICIPAL_MODELS
            if name in near_best.model.values
        )
        status = "selected by observed-block MAE"

    benchmark = scores_90[
        scores_90.profile_id.eq(item.profile_id)
        & scores_90.model.eq("Recent median")
    ].mae
    selected = scores_90[
        scores_90.profile_id.eq(item.profile_id)
        & scores_90.model.eq(chosen)
    ].mae

    benchmark_mae = benchmark.iloc[0] if len(benchmark) else np.nan
    selected_mae = selected.iloc[0] if len(selected) else np.nan
    selection_rows.append(dict(
        profile_id=item.profile_id,
        label=item.label,
        selected_model=chosen,
        selection_status=status,
        selected_mae_90d=selected_mae,
        median_mae_90d=benchmark_mae,
        skill_vs_median_pct=(
            100 * (1 - selected_mae / benchmark_mae)
            if np.isfinite(benchmark_mae) and benchmark_mae > 0
            else np.nan
        ),
    ))

municipal_selection = pd.DataFrame(selection_rows).merge(
    lag_table.loc[
        lag_table.lag_days.eq(28),
        ["profile_id", "correlation", "paired_blocks"]
    ].rename(columns={
        "correlation": "observed_28d_correlation",
        "paired_blocks": "observed_28d_pairs",
    }),
    on="profile_id", how="left",
)

backtest.to_csv(TABLE_DIR / "municipal_pseudo_event_predictions.csv", index=False)
model_scores.to_csv(TABLE_DIR / "municipal_model_scores.csv", index=False)
municipal_selection.to_csv(TABLE_DIR / "municipal_model_selection.csv", index=False)

display(municipal_selection[[
    "label", "selected_model", "selected_mae_90d",
    "median_mae_90d", "skill_vs_median_pct",
    "observed_28d_correlation", "observed_28d_pairs",
    "selection_status",
]].round(2))

if fit_failures:
    display(pd.DataFrame(fit_failures))

Green cells indicate lower 90-day error than the recent-median benchmark; model selection does not establish recovery.

In [ ]:
# Relative 90-day forecast error: green beats the recent-median benchmark.
mae_grid = (
    scores_90.pivot(index="profile_id", columns="model", values="mae")
    .reindex(index=municipal_meta.profile_id, columns=MUNICIPAL_MODELS)
)
relative_mae = (
    mae_grid.div(mae_grid["Recent median"], axis=0)
    .replace([np.inf, -np.inf], np.nan)
)

fig = go.Figure()
fig.add_trace(go.Heatmap(
    x=list(MUNICIPAL_MODELS),
    y=municipal_meta.label,
    z=relative_mae.values,
    colorscale=[
        [0, "#1B7837"], [.5, "#FFFFFF"], [1, "#B2182B"]
    ],
    zmin=0, zmax=2, zmid=1,
    colorbar=dict(
        title=dict(text="MAE /<br>median", font=dict(size=16)),
        tickfont=dict(size=14),
        thickness=18,
    ),
    customdata=mae_grid.values,
    hovertemplate=(
        "%{y}<br>%{x}"
        "<br>MAE %{customdata:.2f} nW cm⁻² sr⁻¹"
        "<br>Relative MAE %{z:.2f}<extra></extra>"
    ),
))
fig.add_trace(go.Scatter(
    x=municipal_selection.selected_model,
    y=municipal_selection.label,
    mode="markers",
    marker=dict(
        symbol="square-open", size=24,
        color=PLOT_TEXT_COLOR,
        line=dict(color=PLOT_TEXT_COLOR, width=2),
    ),
    name="Selected model",
    hoverinfo="skip",
))
fig.update_yaxes(autorange="reversed", tickfont=dict(size=15))
fig.update_xaxes(tickfont=dict(size=14))
style_figure(fig, width=1350, height=720, bottom=115)
fig.update_layout(margin=dict(l=175, r=120, t=45, b=115))
finish_figure(fig, "municipal_bau_backtest")

Selected municipal scenarios and departures retain observation coverage and held-out error information.

In [ ]:
# Selected municipal BAU scenarios and observed departures.
projection_rows = []
selected_errors = backtest.merge(
    municipal_selection[["profile_id", "selected_model"]],
    on="profile_id",
)
selected_errors = selected_errors[
    selected_errors.model.eq(selected_errors.selected_model)
]

for item in municipal_meta.itertuples(index=False):
    g = municipal_series[municipal_series.profile_id.eq(item.profile_id)]
    train = g[g.date.lt(EVENT_DATE)]
    post = g[
        g.date.ge(EVENT_DATE)
        & g.date.lt(EVENT_DATE + pd.Timedelta(days=180))
    ].copy()
    method = municipal_selection.set_index("profile_id").loc[
        item.profile_id, "selected_model"
    ]

    post["selected_model"] = method
    post["expected_ntl"] = municipal_bau(train, len(post), method)
    post["lead_days"] = (post.date - EVENT_DATE).dt.days + 1
    post["expected_low"] = np.nan
    post["expected_high"] = np.nan

    historical = selected_errors[
        selected_errors.profile_id.eq(item.profile_id)
    ]
    for lower_day, upper_day in ((0, 30), (30, 90), (90, 180)):
        errors = historical.loc[
            historical.lead_days.gt(lower_day)
            & historical.lead_days.le(upper_day),
            "error",
        ]
        if len(errors) < 8:
            continue

        low_error, high_error = errors.quantile([.10, .90])
        current = post.lead_days.gt(lower_day) & post.lead_days.le(upper_day)
        post.loc[current, "expected_low"] = np.maximum(
            0, post.loc[current, "expected_ntl"] + low_error
        )
        post.loc[current, "expected_high"] = np.maximum(
            0, post.loc[current, "expected_ntl"] + high_error
        )

    post["difference_ntl"] = post.ntl_rq - post.expected_ntl
    post["difference_pct"] = np.where(
        post.expected_ntl.ge(.10),
        100 * post.difference_ntl / post.expected_ntl,
        np.nan,
    )
    projection_rows.append(post)

municipal_projection = pd.concat(projection_rows, ignore_index=True)
municipal_projection.to_csv(
    TABLE_DIR / "municipal_selected_bau_and_departures.csv",
    index=False,
)

impact_rows = []
for item in municipal_meta.itertuples(index=False):
    profile = municipal_projection[
        municipal_projection.profile_id.eq(item.profile_id)
    ]
    for start_day, end_day in ((0, 30), (30, 90), (90, 180)):
        window = profile[
            profile.lead_days.gt(start_day)
            & profile.lead_days.le(end_day)
        ]
        observed = window[window.ntl_rq.notna()]
        expected_total = observed.expected_ntl.sum()
        comparable = observed[observed.expected_low.notna()]

        impact_rows.append(dict(
            municipality=item.label,
            window=f"{start_day + 1}–{end_day} days",
            selected_model=profile.selected_model.iloc[0],
            fixed_pixels=item.fixed_pixels,
            sc_threshold_pct=SPATIAL_COMPLETENESS_PCT,
            calendar_blocks=len(window),
            observed_blocks=len(observed),
            observed_share_pct=(
                100 * len(observed) / len(window) if len(window) else np.nan
            ),
            median_sc_pct=observed.sc_pct.median(),
            net_deviation_pct=(
                100 * observed.difference_ntl.sum() / expected_total
                if expected_total > 0 else np.nan
            ),
            observed_block_shortfall_pct=(
                100 * np.maximum(
                    observed.expected_ntl - observed.ntl_rq, 0
                ).sum() / expected_total
                if expected_total > 0 else np.nan
            ),
            below_historical_range_blocks=int(
                (comparable.ntl_rq < comparable.expected_low).sum()
            ),
        ))

municipal_impact = pd.DataFrame(impact_rows)
municipal_impact.to_csv(
    TABLE_DIR / "municipal_bau_departure_by_window.csv",
    index=False,
)
display(municipal_impact.round(2))

The municipal small multiples show each observed trajectory against its selected BAU scenario.

In [ ]:
# Municipality-wide G3: observed four-day NTL versus selected BAU.
columns = 4
rows = int(np.ceil(len(municipal_meta) / columns))
fig = make_subplots(
    rows=rows, cols=columns,
    subplot_titles=municipal_meta.label.tolist(),
    horizontal_spacing=.055,
    vertical_spacing=.12,
)

for j, item in enumerate(municipal_meta.itertuples(index=False)):
    row, col = divmod(j, columns)
    row, col = row + 1, col + 1

    observed = municipal_series[
        municipal_series.profile_id.eq(item.profile_id)
        & municipal_series.date.ge(EVENT_DATE - pd.Timedelta(days=365))
        & municipal_series.date.lt(EVENT_DATE + pd.Timedelta(days=180))
    ]
    expected = municipal_projection[
        municipal_projection.profile_id.eq(item.profile_id)
    ]

    fig.add_trace(go.Scatter(
        x=expected.date, y=expected.expected_low,
        mode="lines", line=dict(width=0),
        hoverinfo="skip", showlegend=False,
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=expected.date, y=expected.expected_high,
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(230,106,0,.14)",
        name="Historical forecast-error range",
        showlegend=(j == 0), hoverinfo="skip",
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=expected.date, y=expected.expected_ntl,
        mode="lines",
        line=dict(color="#E66A00", width=2.5),
        name="Selected BAU",
        showlegend=(j == 0),
        hovertemplate="%{x|%d %b %Y}<br>BAU NTL %{y:.2f}<extra></extra>",
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=observed.date, y=observed.ntl_rq,
        mode="lines+markers", connectgaps=False,
        line=dict(color=item.color, width=1.8),
        marker=dict(color=item.color, size=4),
        customdata=observed[["sc_pct", "n_valid", "n_fixed"]],
        name="RQ NTL (4D)",
        showlegend=(j == 0),
        hovertemplate=(
            "%{x|%d %b %Y}<br>RQ NTL %{y:.2f}"
            "<br>SC %{customdata[0]:.0f}%"
            "<br>Pixels %{customdata[1]:.0f}/%{customdata[2]:.0f}"
            "<extra></extra>"
        ),
    ), row=row, col=col)

    fig.add_vline(
        x=EVENT_DATE, line_color=EVENT_COLOR,
        line_dash="dash", line_width=1.3,
        row=row, col=col,
    )
    fig.update_xaxes(
        range=[
            EVENT_DATE - pd.Timedelta(days=365),
            EVENT_DATE + pd.Timedelta(days=180),
        ],
        tickformat="%b %Y", dtick="M6",
        tickfont=dict(size=12),
        row=row, col=col,
    )
    fig.update_yaxes(
        rangemode="tozero",
        tickfont=dict(size=12),
        title_text="NTL (nW cm⁻² sr⁻¹)" if col == 1 else "",
        row=row, col=col,
    )

style_figure(
    fig, width=1800, height=max(950, 340 * rows), bottom=125
)
fig.update_layout(
    legend=dict(
        orientation="h", x=0, y=-.075,
        font=dict(size=16),
    ),
    margin=dict(l=90, r=35, t=70, b=125),
)
fig.update_annotations(font=dict(size=16, color=PLOT_TEXT_COLOR))
finish_figure(fig, "municipal_selected_bau_profiles")

In [ ]:
render_profiles(select_plot_meta("municipality"))

## 6. 5×5 anchors and centre POIs

The same refined STL/impact panel is applied to each 5×5 support; the map identifies the exact reference-qualified area.

In [ ]:
render_profiles(select_plot_meta("kernel5"))

## 7. GHSL settlement classes

Class-wide curves should be read alongside observability and fixed-pixel support. They summarize different settlement morphologies, not interchangeable physical areas.

In [ ]:
render_profiles(select_plot_meta("ghsl"))

## 8. Selected 1×1 Tacloban pixels

The single-pixel example makes the STL fit and projection mechanics visible before the complete selected-site panel.

### BLISS Sagkahan: what can the baseline predict?

STL separates the observed pre-Haiyan NTL into a smooth trend, an assumed 28-day component, and unexplained variation. The projection holds the final pre-event trend level and repeats the fitted 28-day component. It does not use post-Haiyan values to fit the baseline.

The orange line is continuous; breaks in the pink observed line indicate blocks without reliability-qualified observations. The pale band shows the middle 80% of pre-event residual variation, **not a forecast confidence interval**. Individual spikes cannot be assigned reliable future dates. In the pre-event holdout, the recent-median benchmark performed slightly better than this STL projection (MAE 1.00 versus 1.06), so the STL projection is illustrative rather than the preferred impact baseline.

In [ ]:
# BLISS Sagkahan · pre-event STL fit projected after Haiyan

demo_id = "pixel1::BLISS Sagkahan residential"
trend_window = 31
demo_days = 365

record = four_day.loc[
    four_day.profile_id.eq(demo_id)
].sort_values("block")

pre_blocks = np.arange(all_blocks.min(), 0)
baseline = record.set_index("block").ntl_rq.reindex(pre_blocks)

if baseline.notna().sum() < MIN_STL_OBSERVED_BLOCKS:
    raise ValueError("Insufficient observed baseline blocks for BLISS STL.")

pre_dates = EVENT_DATE + pd.to_timedelta(
    pre_blocks * AGGREGATION_DAYS + 1.5, unit="D"
)

# Fill only the pre-event input needed by STL.
filled = baseline.interpolate(
    method="linear", limit_direction="both"
)

result = STL(
    np.log1p(filled.to_numpy()),
    period=STL_PERIOD_BLOCKS,
    seasonal=7,
    trend=trend_window,
    robust=True,
).fit()

trend_pre = np.asarray(result.trend)
seasonal_pre = np.asarray(result.seasonal)

fit_log_pre = trend_pre + seasonal_pre
fitted_pre = np.maximum(0, np.expm1(fit_log_pre))

# Smooth one full 28-day cycle to retain the broad shape of the STL fit.
fit_envelope_pre = (
    pd.Series(fit_log_pre)
    .rolling(
        window=STL_PERIOD_BLOCKS,
        center=True,
        min_periods=1,
    )
    .mean()
    .to_numpy()
)

future_blocks = np.arange(
    int(np.ceil(demo_days / AGGREGATION_DAYS))
)
future_dates = EVENT_DATE + pd.to_timedelta(
    future_blocks * AGGREGATION_DAYS + 1.5, unit="D"
)

# Project the prior year's pre-event STL-fit curve into the
# corresponding dates after Haiyan. Anchor its starting level
# to the final pre-event fitted envelope.
analogue_dates = future_dates - pd.Timedelta(days=365)

pre_day = np.asarray(
    (pre_dates - pre_dates[0]) / pd.Timedelta(days=1),
    dtype=float,
)
analogue_day = np.asarray(
    (analogue_dates - pre_dates[0]) / pd.Timedelta(days=1),
    dtype=float,
)

analogue_fit = np.interp(
    analogue_day, pre_day, fit_envelope_pre
)
expected_log = (
    fit_envelope_pre[-1]
    + analogue_fit
    - analogue_fit[0]
)
expected_ntl = np.maximum(0, np.expm1(expected_log))

# Show the corresponding broad trend shape separately.
analogue_trend = np.interp(
    analogue_day, pre_day, trend_pre
)
trend_future = (
    trend_pre[-1]
    + analogue_trend
    - analogue_trend[0]
)

# Spread around the smoothed pre-event fit is descriptive;
# it is not a prediction interval.
baseline_departure = (
    np.log1p(baseline.to_numpy(dtype=float))
    - fit_envelope_pre
)
valid_departure = baseline_departure[
    np.isfinite(baseline_departure)
]
departure_low, departure_high = np.quantile(
    valid_departure, [.10, .90]
)

range_low = np.maximum(
    0, np.expm1(expected_log + departure_low)
)
range_high = np.maximum(
    0, np.expm1(expected_log + departure_high)
)

recent = baseline[
    baseline.index >= -int(np.ceil(365 / AGGREGATION_DAYS))
].dropna()
if recent.empty:
    recent = baseline.dropna()
median_benchmark = float(recent.median())

post = record.loc[
    record.block.between(0, future_blocks.max())
].copy()
post_blocks = post.block.to_numpy(dtype=int)
post_departure = (
    np.log1p(post.ntl_rq.to_numpy(dtype=float))
    - expected_log[post_blocks]
)

observed_color = "#CC79A7"
fit_color = "#243B5A"
projection_color = "#D55E00"
envelope_color = "#009E73"
benchmark_color = "#81909F"

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=.04,
    row_heights=[.42, .23, .20, .15],
    subplot_titles=[
        "Observed radiance and expected range",
        "STL trend and projected curve",
        "Smoothed STL fit used for projection",
        "Departure from the expected curve",
    ],
)

# Observed radiance and projected expectation.
fig.add_trace(go.Scatter(
    x=future_dates, y=range_low,
    mode="lines",
    line=dict(width=0),
    hoverinfo="skip",
    showlegend=False,
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=future_dates, y=range_high,
    mode="lines",
    line=dict(width=0),
    fill="tonexty",
    fillcolor="rgba(213,94,0,.13)",
    name="Pre-event variation",
    hoverinfo="skip",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=pre_dates, y=fitted_pre,
    mode="lines",
    line=dict(color=fit_color, width=2.1),
    name="Pre-event STL fit",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=pre_dates,
    y=baseline.to_numpy(dtype=float),
    mode="lines+markers",
    connectgaps=False,
    line=dict(color=observed_color, width=1.4),
    marker=dict(color=observed_color, size=3),
    name="Observed RQ NTL",
    legendgroup="observed",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=future_dates, y=expected_ntl,
    mode="lines",
    line=dict(color=projection_color, width=2.8),
    name="Projected STL-fit curve",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=future_dates,
    y=np.full(len(future_dates), median_benchmark),
    mode="lines",
    line=dict(color=benchmark_color, width=1.6),
    name="Recent median",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=post.date, y=post.ntl_rq,
    mode="lines+markers",
    connectgaps=False,
    line=dict(color=observed_color, width=1.7),
    marker=dict(color=observed_color, size=3.5),
    name="Observed RQ NTL",
    legendgroup="observed",
    showlegend=False,
    customdata=post[["sc_pct", "n_valid", "n_fixed"]],
    hovertemplate=(
        "%{x|%d %b %Y}<br>NTL %{y:.2f}"
        "<br>SC %{customdata[0]:.0f}%"
        "<br>Pixels %{customdata[1]:.0f}/%{customdata[2]:.0f}"
        "<extra></extra>"
    ),
), row=1, col=1)

# Trend: the prior-year trend shape is shifted forward.
fig.add_trace(go.Scatter(
    x=pre_dates, y=trend_pre,
    mode="lines",
    line=dict(color=fit_color, width=2.3),
    showlegend=False,
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=future_dates, y=trend_future,
    mode="lines",
    line=dict(color=projection_color, width=2.3),
    showlegend=False,
), row=2, col=1)

# This is the smoothed fitted curve whose shape is projected.
fig.add_trace(go.Scatter(
    x=pre_dates, y=fit_envelope_pre,
    mode="lines",
    line=dict(color=envelope_color, width=2.2),
    showlegend=False,
), row=3, col=1)

fig.add_trace(go.Scatter(
    x=future_dates, y=expected_log,
    mode="lines",
    line=dict(color=projection_color, width=2.2),
    showlegend=False,
), row=3, col=1)

# Compare observed departures with pre-event departures
# measured against the same smoothed-fit definition.
fig.add_trace(go.Scatter(
    x=pre_dates, y=baseline_departure,
    mode="markers",
    marker=dict(
        color=benchmark_color, size=4, opacity=.75
    ),
    showlegend=False,
), row=4, col=1)

fig.add_trace(go.Scatter(
    x=post.date, y=post_departure,
    mode="markers",
    marker=dict(
        color=observed_color, size=5, opacity=.9
    ),
    showlegend=False,
), row=4, col=1)

fig.add_hline(
    y=0, line_color=PLOT_GRID_COLOR, line_width=1,
    row=4, col=1,
)

for row in range(1, 5):
    fig.add_vline(
        x=EVENT_DATE,
        line_color=EVENT_COLOR,
        line_dash="dash",
        line_width=1.6,
        row=row, col=1,
    )
    fig.update_xaxes(
        range=[pre_dates.min(), future_dates.max()],
        dtick="M3",
        tickformat="%b %Y",
        tickfont=dict(size=15),
        showticklabels=(row == 4),
        row=row, col=1,
    )
    fig.update_yaxes(
        tickfont=dict(size=15),
        row=row, col=1,
    )

fig.update_yaxes(
    title_text="NTL (nW cm⁻² sr⁻¹)", row=1, col=1
)
fig.update_yaxes(
    title_text="log(1 + NTL)", row=2, col=1
)
fig.update_yaxes(
    title_text="log(1 + NTL)", row=3, col=1
)
fig.update_yaxes(
    title_text="log departure", row=4, col=1
)

style_figure(fig, width=1500, height=1050, bottom=130)
fig.update_layout(
    margin=dict(l=105, r=35, t=55, b=130),
    legend=dict(
        orientation="h",
        x=0, y=-.09,
        font=dict(size=20),
    ),
)
fig.update_annotations(
    font=dict(size=18, color=PLOT_TEXT_COLOR)
)
finish_figure(fig, "00_bliss_stl_projection_refined")

The BLISS holdout varies only the STL trend smoother; the 28-day diagnostic period and observed qualification remain fixed.

In [ ]:
# BLISS only: test STL trend-smoothing lengths on pre-Haiyan holdouts.
# Each block is four days. This tests smoothing, not a linear trend extension.

name = "pixel1::BLISS Sagkahan residential"
blocks = np.arange(all_blocks.min(), 0)

series = (
    four_day.loc[four_day.profile_id.eq(name)]
    .set_index("block")
    .ntl_rq
    .reindex(blocks)
)

rows = []

for trend_window in (15, 31, 45, 61):  # 60, 124, 180, 244 days
    for holdout_days in (60, 120, 180):
        cutoff = -int(np.ceil(holdout_days / AGGREGATION_DAYS))
        train = series[series.index < cutoff]
        test = series[series.index >= cutoff]

        valid_test = test.notna().to_numpy()
        if train.notna().sum() < 24 or valid_test.sum() < 8:
            continue

        # Interpolation is confined to the training input for STL.
        filled = train.interpolate(
            method="linear", limit_direction="both"
        )
        result = STL(
            np.log1p(filled.to_numpy()),
            period=7,              # Existing 28-day hypothesis
            seasonal=7,
            trend=trend_window,
            robust=True,
        ).fit()

        # Forecast a constant final trend level; repeat the fitted component.
        train_blocks = train.index.to_numpy()
        seasonal = np.asarray(result.seasonal)
        phase = np.array([
            np.median(seasonal[train_blocks % 7 == k])
            for k in range(7)
        ])
        trend_level = np.median(np.asarray(result.trend)[-7:])

        predicted = np.maximum(
            0,
            np.expm1(
                trend_level + phase[test.index.to_numpy() % 7]
            ),
        )

        recent_train = train.iloc[-91:]  # Approximately one year
        flat_level = recent_train.median()
        if not np.isfinite(flat_level):
            flat_level = train.median()

        actual = test.to_numpy(dtype=float)
        rows.append(dict(
            trend_blocks=trend_window,
            trend_days=trend_window * AGGREGATION_DAYS,
            holdout_days=holdout_days,
            observed_holdout_blocks=int(valid_test.sum()),
            stl_mae=np.mean(
                np.abs(actual[valid_test] - predicted[valid_test])
            ),
            flat_mae=np.mean(
                np.abs(actual[valid_test] - flat_level)
            ),
        ))

window_check = pd.DataFrame(rows)
display(window_check.round(2))

window_summary = (
    window_check.groupby(["trend_blocks", "trend_days"])
    [["stl_mae", "flat_mae"]]
    .median()
    .sort_values("stl_mae")
)
display(window_summary.round(2))

The selected-pixel small multiples project the pre-event STL fit after Haiyan. The curves are illustrative expectations; gappy post-event observations remain distinct.

In [ ]:
# Twelve selected 1×1 sites · pre-event STL-fit projection
# Pixel 11 is excluded.

site_names = [
    "Airport",
    "Port",
    "Main-road vicinity",
    "BLISS Sagkahan residential",
    "Kassel City residential",
    "Rizal Plaza",
    "Astrodome",
    "Robinsons Place",
    "Pixel 23",
    "Pixel 21",
    "Pixel 13",
    "Pixel 12",
]

trend_window = 31
projection_days = 365

pre_blocks = np.arange(all_blocks.min(), 0)
pre_dates = EVENT_DATE + pd.to_timedelta(
    pre_blocks * AGGREGATION_DAYS + 1.5, unit="D"
)

future_blocks = np.arange(
    int(np.ceil(projection_days / AGGREGATION_DAYS))
)
future_dates = EVENT_DATE + pd.to_timedelta(
    future_blocks * AGGREGATION_DAYS + 1.5, unit="D"
)

# Dates in the preceding year supply the projected curve's shape.
analogue_dates = future_dates - pd.Timedelta(days=365)
pre_day = np.asarray(
    (pre_dates - pre_dates[0]) / pd.Timedelta(days=1),
    dtype=float,
)
analogue_day = np.asarray(
    (analogue_dates - pre_dates[0]) / pd.Timedelta(days=1),
    dtype=float,
)

meta_by_id = profile_meta.set_index("profile_id")

# Keep one vertical scale across the eight named locations.
named_ids = [f"pixel1::{name}" for name in site_names[:8]]
named_values = four_day.loc[
    four_day.profile_id.isin(named_ids)
    & four_day.block.le(future_blocks.max()),
    "ntl_rq",
].dropna()
named_ymax = (
    max(12., np.ceil(named_values.max() * 1.08))
    if len(named_values) else 12.
)

fig = make_subplots(
    rows=3, cols=4,
    horizontal_spacing=.055,
    vertical_spacing=.105,
    subplot_titles=site_names,
)

for i, name in enumerate(site_names):
    row, col = divmod(i, 4)
    row += 1
    col += 1

    profile_id = f"pixel1::{name}"
    if profile_id not in meta_by_id.index:
        print(f"No eligible 1×1 profile: {name}")
        continue

    record = four_day.loc[
        four_day.profile_id.eq(profile_id)
    ].sort_values("block")

    baseline = (
        record.set_index("block")
        .ntl_rq.reindex(pre_blocks)
    )
    if baseline.notna().sum() < MIN_STL_OBSERVED_BLOCKS:
        print(f"Insufficient observed baseline for {name}")
        continue

    # Fill only the pre-event input required by STL.
    filled = baseline.interpolate(
        method="linear", limit_direction="both"
    )
    result = STL(
        np.log1p(filled.to_numpy()),
        period=STL_PERIOD_BLOCKS,
        seasonal=7,
        trend=trend_window,
        robust=True,
    ).fit()

    fit_log_pre = (
        np.asarray(result.trend)
        + np.asarray(result.seasonal)
    )
    fitted_pre = np.maximum(
        0, np.expm1(fit_log_pre)
    )

    # Remove the rapid 28-day oscillations while retaining
    # the broader rises and falls of the fitted curve.
    fit_envelope_pre = (
        pd.Series(fit_log_pre)
        .rolling(
            window=STL_PERIOD_BLOCKS,
            center=True,
            min_periods=1,
        )
        .mean()
        .to_numpy()
    )

    # Shift the preceding year's fitted shape forward,
    # anchored to the final pre-event fitted level.
    analogue_fit = np.interp(
        analogue_day, pre_day, fit_envelope_pre
    )
    expected_log = (
        fit_envelope_pre[-1]
        + analogue_fit
        - analogue_fit[0]
    )
    expected_ntl = np.maximum(
        0, np.expm1(expected_log)
    )

    # Descriptive pre-event spread around the same
    # smoothed fit; not a prediction interval.
    baseline_departure = (
        np.log1p(baseline.to_numpy(dtype=float))
        - fit_envelope_pre
    )
    valid_departure = baseline_departure[
        np.isfinite(baseline_departure)
    ]
    departure_low, departure_high = np.quantile(
        valid_departure, [.10, .90]
    )
    range_low = np.maximum(
        0, np.expm1(expected_log + departure_low)
    )
    range_high = np.maximum(
        0, np.expm1(expected_log + departure_high)
    )

    # Observed RQ median during the final pre-event year.
    recent = baseline[
        baseline.index >= -int(
            np.ceil(365 / AGGREGATION_DAYS)
        )
    ].dropna()
    if recent.empty:
        recent = baseline.dropna()
    recent_median = float(recent.median())

    post = record.loc[
        record.block.between(0, future_blocks.max())
    ]

    observed_color = meta_by_id.loc[
        profile_id, "color"
    ]

    # Darken pale GHSL colours for display on white.
    if name == "Pixel 21":
        observed_color = "#B49B00"
    elif name == "Pixel 12":
        observed_color = "#789A36"

    # Draw the variation range behind the trajectories.
    fig.add_trace(go.Scatter(
        x=future_dates,
        y=range_low,
        mode="lines",
        line=dict(width=0),
        hoverinfo="skip",
        showlegend=False,
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=future_dates,
        y=range_high,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor="rgba(213,94,0,.11)",
        name="Pre-event variation",
        showlegend=(i == 0),
        hoverinfo="skip",
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=pre_dates,
        y=fitted_pre,
        mode="lines",
        line=dict(color="#243B5A", width=1.8),
        name="Pre-event STL fit",
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=future_dates,
        y=expected_ntl,
        mode="lines",
        line=dict(color="#D55E00", width=2.5),
        name="Projected STL-fit curve",
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=future_dates,
        y=np.full(len(future_dates), recent_median),
        mode="lines",
        line=dict(
            color="#81909F",
            width=1.7,
            dash="dot",
        ),
        name="Recent pre-event median",
        showlegend=(i == 0),
        hovertemplate=(
            "Pre-event 365-day median: %{y:.2f}"
            "<extra></extra>"
        ),
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=post.date,
        y=post.ntl_rq,
        mode="lines+markers",
        connectgaps=False,
        line=dict(color=observed_color, width=1.5),
        marker=dict(color=observed_color, size=3),
        name="Observed RQ NTL",
        legendgroup="observed",
        showlegend=(i == 0),
        customdata=post[["sc_pct", "n_valid", "n_fixed"]],
        hovertemplate=(
            "%{x|%d %b %Y}<br>NTL %{y:.2f}"
            "<br>SC %{customdata[0]:.0f}%"
            "<br>Pixels %{customdata[1]:.0f}/"
            "%{customdata[2]:.0f}"
            "<extra></extra>"
        ),
    ), row=row, col=col)

    fig.add_vline(
        x=EVENT_DATE,
        line_color=EVENT_COLOR,
        line_dash="dash",
        line_width=1.4,
        row=row, col=col,
    )

    if i < 8:
        ymax = named_ymax
    else:
        shown = np.concatenate([
            baseline.to_numpy(dtype=float),
            post.ntl_rq.to_numpy(dtype=float),
            range_high,
            [recent_median],
        ])
        shown = shown[np.isfinite(shown)]
        ymax = max(
            .5,
            np.ceil(shown.max() * 1.08 * 2) / 2,
        )

    fig.update_yaxes(
        range=[0, ymax],
        nticks=5,
        tickfont=dict(size=13),
        title_text=(
            "NTL (nW cm⁻² sr⁻¹)" if col == 1 else ""
        ),
        row=row, col=col,
    )
    fig.update_xaxes(
        range=[pre_dates.min(), future_dates.max()],
        dtick="M6",
        tickformat="%b %Y",
        tickfont=dict(size=13),
        showticklabels=(row == 3),
        row=row, col=col,
    )

# One legend entry for the event marker.
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode="lines",
    line=dict(
        color=EVENT_COLOR,
        width=1.4,
        dash="dash",
    ),
    name="Haiyan",
), row=1, col=1)

style_figure(
    fig,
    width=1600,
    height=900,
    bottom=145,
)
fig.update_layout(
    margin=dict(l=115, r=40, t=70, b=145),
    legend=dict(
        orientation="h",
        x=0,
        y=-.075,
        font=dict(size=16),
    ),
)
fig.update_annotations(
    font=dict(size=18, color=PLOT_TEXT_COLOR)
)

finish_figure(
    fig,
    "00_selected_pixels_stl_projection_4x3",
)


### Selected-pixel nighttime lights after Haiyan

The navy line is each pixel’s pre-Haiyan STL fit. The orange line projects the broad shape of that fitted curve using the preceding year, anchored at the final pre-event level. The dotted line shows the median observed radiance during the final pre-event year. Orange shading represents the spread of pre-event observations around the smoothed fit; it is **not a prediction interval**.

Many named urban locations show an immediate decline in reliability-qualified NTL after Haiyan. The Port, main-road vicinity, Rizal Plaza, and Robinsons Place subsequently move closer to their projected levels, although their trajectories remain irregular. The Airport and the two residential pixels show more persistent departures. Among the comparison pixels, classes 21 and 12 also spend substantial time below their projected curves; the dimmer class 23 and 13 pixels are harder to interpret because modest absolute changes are large relative to their baseline brightness.

These patterns suggest that the shock and subsequent trajectory differ across ground conditions, even within the same GHSL class. They do **not** establish recovery on their own: gaps indicate insufficient observability, not zero radiance, and occasional bright observations do not demonstrate sustained return to baseline.

The orange curves are illustrative business-as-usual analogues, not validated counterfactual forecasts. Impact and recovery estimates should therefore be checked against the recent-median benchmark, retained observation coverage, and the duration of any sustained departure.

In [ ]:
render_profiles(select_plot_meta("pixel1"))

In [ ]:
render_profiles(select_plot_meta("pixel8"))

## 9. Cross-scale evidence and impact synthesis

These tables and figures compare recurrence, model support, magnitude, duration, and observability across the four study scales and regional overview.

### Observed recurrence across scales

The heatmap checks correlations at **20–80-day and 336–392-day lags** using observed baseline pairs only. A strong cell does not establish a cycle. `observed_cycle_peak_days` is reported only when the short-lag maximum correlation reaches 0.2; the annual-lag statistic is exploratory because fewer than two complete pre-Haiyan years are available. The STL seasonal-strength index describes the fraction of detrended fitted variation assigned to the imposed 28-day component, and should be read alongside baseline interpolation.

| Output | What it tells you |
|---|---|
| Lag heatmaps | Whether *observed* NTL values resemble values 20–80 days later, or roughly a year later. The paired-block count shows how much evidence supports each correlation. |
| Seasonal-strength plots | How much variation the STL model assigns to its **specified 28-day component**, alongside how much of the baseline was actually observed. A high value alone does not establish a real 28-day cycle. |
| Evidence tables | Which apparent cycles have enough observed blocks and paired dates to be interpretable at each spatial scale. |

In [ ]:
seasonality = model_audit.merge(
    profile_meta[["profile_id", "color"]],
    on="profile_id",
)
seasonality["observed_share_pct"] = (
    100 - seasonality.baseline_interpolated_pct
)

for scale, scale_title in zip(scale_order, scale_titles):
    group = seasonality[
        seasonality.scale.eq(scale)
    ].copy()
    if group.empty:
        continue

    display_colors = group.color.replace({
        "#ffff00": "#B49B00",
        "#cdf57a": "#789A36",
        "#abcd66": "#789A36",
    })

    fig = make_subplots(
        rows=1,
        cols=2,
        horizontal_spacing=.14,
        subplot_titles=[
            "Fitted 28-day component",
            "Recurrence in observed blocks",
        ],
    )

    fig.add_trace(go.Scatter(
        x=group.observed_share_pct,
        y=group.seasonal_strength,
        mode="markers",
        marker=dict(
            size=12,
            color=display_colors,
            line=dict(color="white", width=1),
        ),
        text=group.label,
        customdata=group[
            ["baseline_observed_blocks"]
        ],
        hovertemplate=(
            "%{text}"
            "<br>STL strength %{y:.2f}"
            "<br>Observed baseline %{x:.1f}%"
            "<br>Observed blocks %{customdata[0]:.0f}"
            "<extra></extra>"
        ),
        showlegend=False,
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=group.observed_cycle_peak_days,
        y=group.peak_lag_correlation,
        mode="markers",
        marker=dict(
            size=12,
            color=display_colors,
            line=dict(color="white", width=1),
        ),
        text=group.label,
        customdata=group[["peak_lag_pairs"]],
        hovertemplate=(
            "%{text}"
            "<br>Peak lag %{x:.0f} days"
            "<br>Correlation %{y:.2f}"
            "<br>Observed pairs %{customdata[0]:.0f}"
            "<extra></extra>"
        ),
        showlegend=False,
    ), row=1, col=2)

    fig.add_vline(
        x=28,
        line_color=BASELINE_COLOR,
        line_dash="dot",
        line_width=1,
        row=1,
        col=2,
    )
    fig.add_hline(
        y=.2,
        line_color=BASELINE_COLOR,
        line_dash="dot",
        line_width=1,
        row=1,
        col=2,
    )

    style_figure(
        fig,
        width=1450,
        height=630,
        bottom=105,
    )
    fig.update_layout(
        title=dict(
            text=scale_title,
            x=.03,
            font=dict(size=20),
        ),
    )
    fig.update_xaxes(
        title_text="Observed baseline blocks (%)",
        range=[0, 100],
        row=1,
        col=1,
    )
    fig.update_yaxes(
        title_text="STL seasonal strength",
        range=[0, 1],
        row=1,
        col=1,
    )
    fig.update_xaxes(
        title_text="Peak observed lag (days)",
        range=[18, 82],
        dtick=8,
        row=1,
        col=2,
    )
    fig.update_yaxes(
        title_text="Observed-pair correlation",
        row=1,
        col=2,
    )
    finish_figure(
        fig,
        f"05_seasonality_support_{scale}",
    )

In [ ]:
table_columns = [
    "label",
    "fit_status",
    "baseline_observed_blocks",
    "baseline_calendar_blocks",
    "observed_share_pct",
    "seasonal_strength",
    "observed_cycle_peak_days",
    "peak_lag_correlation",
    "peak_lag_pairs",
    "annual_lag_peak_days",
    "annual_lag_correlation",
    "annual_lag_pairs",
]

for scale, scale_title in zip(scale_order, scale_titles):
    table = (
        seasonality.loc[
            seasonality.scale.eq(scale),
            table_columns,
        ]
        .sort_values("label")
        .reset_index(drop=True)
    )

    print(scale_title)
    display(table.round(2))

In [ ]:
ordered = profile_meta.set_index("profile_id").loc[
    [pid for scale in scale_order
     for pid in profile_meta.loc[profile_meta.scale.eq(scale), "profile_id"]]
]
lag_grid = lag_table.pivot(index="profile_id", columns="lag_days", values="correlation")
pair_grid = lag_table.pivot(index="profile_id", columns="lag_days", values="paired_blocks")
lag_grid = lag_grid.reindex(ordered.index)
pair_grid = pair_grid.reindex(ordered.index)
fig = make_subplots(rows=1, cols=2, horizontal_spacing=.08,
                    column_widths=[.6, .4],
                    subplot_titles=["20–80-day lags", "336–392-day lags"])
for col, selected in ((1, lag_grid.columns[lag_grid.columns <= 80]),
                      (2, lag_grid.columns[lag_grid.columns >= 336])):
    fig.add_trace(go.Heatmap(
        x=selected, y=ordered.label, z=lag_grid[selected].values,
        colorscale="RdBu", reversescale=True, zmin=-.6, zmax=.6, zmid=0,
        showscale=(col == 2),
        colorbar=dict(title="Observed<br>correlation", thickness=15),
        customdata=pair_grid[selected].values,
        hovertemplate=("%{y}<br>Lag %{x} days<br>Correlation %{z:.2f}"
                       "<br>Observed pairs %{customdata:.0f}<extra></extra>"),
    ), row=1, col=col)
    fig.update_xaxes(title_text="Observed lag (days)",
                     tickvals=[28, 56, 80] if col == 1 else [336, 364, 392],
                     row=1, col=col)
    fig.update_yaxes(autorange="reversed", tickfont=dict(size=11),
                     showticklabels=(col == 1), row=1, col=col)
style_figure(fig, width=1500, height=max(1000, 35 * len(ordered)), bottom=85)
finish_figure(fig, "04_observed_baseline_lag_correlations")

seasonality = model_audit.merge(
    profile_meta[["profile_id", "color"]], on="profile_id"
)
fig = make_subplots(rows=1, cols=2, horizontal_spacing=.14,
                    subplot_titles=["28-day component", "Observed recurrence"])
for scale in scale_order:
    g = seasonality[seasonality.scale.eq(scale)]
    fig.add_trace(go.Scatter(
        x=g.baseline_interpolated_pct, y=g.seasonal_strength,
        mode="markers", name=scale_titles[scale_order.index(scale)],
        marker=dict(size=11, color=g.color, line=dict(color="white", width=1)),
        text=g.label,
        hovertemplate=("%{text}<br>Seasonal strength %{y:.2f}"
                       "<br>Baseline interpolated %{x:.1f}%<extra></extra>"),
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=g.observed_cycle_peak_days, y=g.peak_lag_correlation,
        mode="markers", showlegend=False,
        marker=dict(size=11, color=g.color, line=dict(color="white", width=1)),
        text=g.label,
        hovertemplate=("%{text}<br>Recurrence %{x:.0f} days"
                       "<br>Observed correlation %{y:.2f}<extra></extra>"),
    ), row=1, col=2)
style_figure(fig, width=1450, height=650, bottom=125)
fig.update_xaxes(title_text="Baseline blocks interpolated (%)", row=1, col=1)
fig.update_yaxes(title_text="STL seasonal strength", range=[0, 1], row=1, col=1)
fig.update_xaxes(title_text="Peak observed lag (days)", range=[18, 82], dtick=8, row=1, col=2)
fig.update_yaxes(title_text="Observed-pair correlation", row=1, col=2)
finish_figure(fig, "05_seasonality_and_observation_support")
display(seasonality[["scale", "label", "baseline_observed_blocks",
                     "baseline_interpolated_pct", "seasonal_strength",
                     "observed_cycle_peak_days", "peak_lag_correlation",
                     "annual_lag_peak_days", "annual_lag_pairs",
                     "annual_lag_correlation"]].round(2))

In [ ]:
primary = impact[impact.window_days.eq(PRIMARY_IMPACT_DAYS)].copy()
fig = make_subplots(
    rows=4, cols=2, horizontal_spacing=.19, vertical_spacing=.085,
    subplot_titles=[title + " · worst observed drop" if side == 0
                    else title + " · observed-block shortfall"
                    for title in scale_titles for side in (0, 1)],
)
for row, scale in enumerate(scale_order, start=1):
    g = primary[primary.scale.eq(scale)].copy()
    g["marker_size"] = 7 + 13 * g.observable_share.fillna(0)
    for col, measure in ((1, "worst_deviation_pct"),
                         (2, "observed_block_shortfall_pct")):
        fig.add_trace(go.Scatter(
            x=g[measure], y=g.label, mode="markers",
            marker=dict(size=g.marker_size, color=g.color,
                        line=dict(color=PLOT_TEXT_COLOR, width=.6)),
            showlegend=False,
            customdata=g[["observed_blocks", "calendar_blocks", "fixed_pixels"]],
            hovertemplate=("%{y}<br>%{x:.1f}%"
                           "<br>Observed blocks %{customdata[0]:.0f}/%{customdata[1]:.0f}"
                           "<br>Fixed pixels %{customdata[2]:.0f}<extra></extra>"),
        ), row=row, col=col)
        fig.update_yaxes(autorange="reversed", tickfont=dict(size=12), row=row, col=col)
        fig.update_xaxes(tickfont=dict(size=12), row=row, col=col)
    fig.add_vline(x=0, line_color=BASELINE_COLOR, line_width=1, row=row, col=1)
fig.update_xaxes(title_text="Observed − expected (%)", row=4, col=1)
fig.update_xaxes(title_text="Positive shortfall / expected NTL (%)", row=4, col=2)
style_figure(fig, width=1600, height=1350, bottom=95)
finish_figure(fig, "06_impact_by_spatial_scale")

paired = primary[primary.profile_id.str.startswith(("municipality::", "kernel5::"))].assign(
    municipality=lambda d: d.profile_id.str.split("::", regex=False).str[-1]
).pivot(index="municipality", columns="scale", values="observed_block_shortfall_pct")
paired.to_csv(TABLE_DIR / "municipality_vs_5x5_shortfall.csv")
fig = go.Figure()
for name in STUDY_AREAS:
    if name not in paired.index or paired.loc[name].isna().any():
        continue
    fig.add_trace(go.Scatter(
        x=[paired.loc[name, "municipality"]], y=[paired.loc[name, "kernel5"]],
        mode="markers+text", text=[name], textposition="top center",
        marker=dict(color=AREA_COLORS[name], size=11, line=dict(color="white", width=1)),
        name=name, showlegend=False,
    ))
fig.add_shape(type="line", x0=0, y0=0, x1=100, y1=100,
              line=dict(color=BASELINE_COLOR, dash="dot", width=1))
style_figure(fig, width=950, height=780, bottom=85)
fig.update_xaxes(title_text="Municipality G3 shortfall (%)", range=[0, 100])
fig.update_yaxes(title_text="Municipal 5×5 shortfall (%)", range=[0, 100])
finish_figure(fig, "07_municipality_vs_5x5_shortfall")

### Post-Haiyan departures

The first 180 days are the primary descriptive comparison. **Worst drop** is the most negative observed four-day difference relative to expected NTL. **Observed-block shortfall** sums positive expected-minus-observed radiance over comparable blocks. Both depend on the baseline scenario. Point area reflects the share of post-event calendar blocks actually observed; the accompanying table should be used before interpreting a deep or shallow departure.

### Reading the departures

Read the observability heatmap and model audit **before** comparing impact metrics. A drop supported by many observed blocks and a stable pre-event model is more interpretable than one surrounded by missing blocks. An uncertain 28-day recurrence is not evidence of annual seasonality. The 365-day scenario is deliberately shown with a flat-trend alternative and an empirical residual range; disagreement between them indicates model dependence. The long observed record is retained for later recovery analysis, but this notebook does not extend a short pre-event linear trend across the entire decade.

The exported CSVs distinguish observed NTL, model inputs, scenarios, and comparable post-event blocks. `impact_metrics_by_window.csv` contains the 60-, 180-, and 365-day summaries and interpretability flags. No missing post-event value is converted to zero.

## 10. Preserved alternative panel layouts

The earlier refined layouts remain available for design comparison. They use separate filenames so they cannot overwrite the primary figures above.

In [ ]:
show_profiles = {
    "municipality::Tacloban",
    "centre5::Tacloban",
    "ghsl::30",
    "pixel1::Airport",
}

plot_start = dates.min()
plot_end = min(
    dates.max(),
    EVENT_DATE + pd.Timedelta(days=FORECAST_DAYS),
)

# Existing 60-day pre-Haiyan pixel reference.
reference_grid = np.full(land.shape, np.nan, dtype=float)
reference_grid.ravel()[
    land_pixels[reference_ok]
] = ntl0[reference_ok]

# GHSL display includes water.
ghsl_codes = [10, 11, 12, 13, 21, 22, 23, 30]
ghsl_colors = [
    "#BBD7EA",
    *[GHSL_CLASS_COLORS[code] for code in ghsl_codes[1:]],
]
ghsl_grid = np.full(land.shape, np.nan)
for index, code in enumerate(ghsl_codes):
    ghsl_grid[ghsl == code] = index

ghsl_scale = [
    [position, color]
    for index, color in enumerate(ghsl_colors)
    for position in (
        index / len(ghsl_codes),
        (index + 1) / len(ghsl_codes),
    )
]

dx = float(abs(a2.x.values[1] - a2.x.values[0]))
dy = float(abs(a2.y.values[1] - a2.y.values[0]))

# Municipal borders, already in the NTL raster CRS.
boundary_x, boundary_y = [], []
for geometry in region.geometry:
    edge = geometry.boundary
    segments = edge.geoms if hasattr(edge, "geoms") else [edge]
    for segment in segments:
        xx, yy = segment.xy
        boundary_x.extend([*xx, None])
        boundary_y.extend([*yy, None])

# Same Haiyan path used by the earlier recovery notebooks.
track_path = (
    DATA_DIR
    / "yolanda-path-line-"
    / "Yolanda Path Line.shp"
)
track_x, track_y = [], []

if track_path.exists():
    haiyan_track = gpd.read_file(
        track_path
    ).to_crs(a2.rio.crs)

    for geometry in haiyan_track.geometry.dropna():
        segments = (
            geometry.geoms
            if hasattr(geometry, "geoms")
            else [geometry]
        )
        for segment in segments:
            if hasattr(segment, "xy"):
                xx, yy = segment.xy
                track_x.extend([*xx, None])
                track_y.extend([*yy, None])
else:
    print(
        "Haiyan path shapefile not found; "
        "the maps will omit the path."
    )

sc_colorscale = [
    [0.00, "#F3F5F8"],
    [0.10, "#DCEEDD"],
    [0.50, "#75BD83"],
    [1.00, "#006B2D"],
]

saved_count = 0

# for item in tqdm(
#     profile_meta.itertuples(),
#     total=len(profile_meta),
#     desc="STL figures",
# ):

preview = profile_meta[
    profile_meta.profile_id.eq("municipality::Tacloban")
]

for item in tqdm(
    preview.itertuples(),
    total=len(preview),
    desc="Plot preview",
):
    
    pid = item.profile_id

    fit = stl_components[
        stl_components.profile_id.eq(pid)
    ].sort_values("date")
    if fit.empty:
        continue

    obs = four_day[
        four_day.profile_id.eq(pid)
    ].sort_values("date")
    obs_short = obs[obs.date.le(plot_end)].copy()

    future_dates = pd.DatetimeIndex(
        forecast.loc[
            forecast.profile_id.eq(pid)
            & forecast.date.le(plot_end),
            "date",
        ].sort_values()
    )
    if len(future_dates) == 0:
        continue

    display_color = {
        "#ffff00": "#B49B00",
        "#cdf57a": "#789A36",
        "#abcd66": "#789A36",
    }.get(str(item.color).lower(), item.color)

    # ------------------------------------------------------------
    # Curved projection of the pre-event STL fit
    # ------------------------------------------------------------

    pre_dates = pd.DatetimeIndex(fit.date)
    fit_log = (
        fit.trend_log.to_numpy(dtype=float)
        + fit.seasonal_log.to_numpy(dtype=float)
    )

    fit_envelope = (
        pd.Series(fit_log)
        .rolling(
            window=STL_PERIOD_BLOCKS,
            center=True,
            min_periods=1,
        )
        .mean()
        .to_numpy()
    )

    analogue_dates = (
        future_dates - pd.Timedelta(days=365)
    )
    pre_day = np.asarray(
        (pre_dates - pre_dates[0])
        / pd.Timedelta(days=1),
        dtype=float,
    )
    analogue_day = np.asarray(
        (analogue_dates - pre_dates[0])
        / pd.Timedelta(days=1),
        dtype=float,
    )

    prior_year_curve = np.interp(
        analogue_day,
        pre_day,
        fit_envelope,
    )
    expected_log = (
        fit_envelope[-1]
        + prior_year_curve
        - prior_year_curve[0]
    )
    expected_ntl = np.maximum(
        0,
        np.expm1(expected_log),
    )

    baseline_departure = (
        np.log1p(
            fit.observed_ntl.to_numpy(dtype=float)
        )
        - fit_envelope
    )
    valid_departure = baseline_departure[
        np.isfinite(baseline_departure)
    ]
    departure_low, departure_high = np.quantile(
        valid_departure,
        [.10, .90],
    )

    range_low = np.maximum(
        0,
        np.expm1(expected_log + departure_low),
    )
    range_high = np.maximum(
        0,
        np.expm1(expected_log + departure_high),
    )

    projected = pd.DataFrame({
        "date": future_dates,
        "expected_ntl": expected_ntl,
    })
    departure = (
        obs_short.loc[
            obs_short.date.ge(EVENT_DATE),
            ["date", "ntl_rq"],
        ]
        .merge(projected, on="date", how="left")
    )
    departure["difference_ntl"] = (
        departure.ntl_rq
        - departure.expected_ntl
    )

    recent = obs.loc[
        obs.date.ge(
            EVENT_DATE - pd.Timedelta(days=365)
        )
        & obs.date.lt(EVENT_DATE),
        "ntl_rq",
    ].dropna()

    # ------------------------------------------------------------
    # Left: SC, NTL, STL + departure.
    # Right: equally sized GHSL and NTL maps.
    # ------------------------------------------------------------

    fig = make_subplots(
        rows=6,
        cols=2,
        specs=[
            [{}, {"rowspan": 3}],
            [{"rowspan": 3}, None],
            [None, None],
            [None, {"rowspan": 3}],
            [{"rowspan": 2, "secondary_y": True}, None],
            [None, None],
        ],
        column_widths=[.70, .30],
        row_heights=[
            .10, .19, .19,
            .19, .165, .165,
        ],
        horizontal_spacing=.07,
        vertical_spacing=.028,
    )

    left_domain = fig.layout.xaxis.domain
    sc_domain = fig.layout.yaxis.domain
    ntl_domain = fig.layout.yaxis3.domain
    components_domain = fig.layout.yaxis5.domain
    ghsl_map_domain = fig.layout.yaxis2.domain
    ntl_map_domain = fig.layout.yaxis4.domain

    # ------------------------------------------------------------
    # Left, top: aligned green SC strip
    # ------------------------------------------------------------

    fig.add_trace(go.Heatmap(
        x=obs_short.date,
        y=["SC"],
        z=obs_short.sc_pct.to_numpy(
            dtype=float
        )[None, :],
        colorscale=sc_colorscale,
        zmin=0,
        zmax=100,
        hoverongaps=False,
        showscale=True,
        colorbar=dict(
            title="SC (%)",
            x=left_domain[1] + .008,
            y=sum(sc_domain) / 2,
            len=sc_domain[1] - sc_domain[0],
            thickness=10,
            tickvals=[0, 50, 100],
            tickfont=dict(size=10),
        ),
        hovertemplate=(
            "%{x|%d %b %Y}"
            "<br>Spatial completeness %{z:.0f}%"
            "<extra></extra>"
        ),
    ), row=1, col=1)

    # ------------------------------------------------------------
    # Left, middle: observed NTL and curved projection
    # ------------------------------------------------------------

    fig.add_trace(go.Scatter(
        x=future_dates,
        y=range_low,
        mode="lines",
        line=dict(width=0),
        hoverinfo="skip",
        showlegend=False,
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=future_dates,
        y=range_high,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor="rgba(213,94,0,.12)",
        name="Pre-event variation",
        legend="legend2",
        hoverinfo="skip",
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=fit.date,
        y=fit.fitted_ntl,
        mode="lines",
        line=dict(
            color="#243B5A",
            width=2,
        ),
        name="Pre-event STL fit",
        legend="legend2",
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=future_dates,
        y=expected_ntl,
        mode="lines",
        line=dict(
            color="#D55E00",
            width=2.7,
        ),
        name="Projected STL-fit curve",
        legend="legend2",
    ), row=2, col=1)

    if not recent.empty:
        fig.add_trace(go.Scatter(
            x=future_dates,
            y=np.full(
                len(future_dates),
                float(recent.median()),
            ),
            mode="lines",
            line=dict(
                color="#81909F",
                width=1.5,
                dash="dot",
            ),
            name="Recent pre-event median",
            legend="legend2",
        ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=obs_short.date,
        y=obs_short.ntl_rq,
        mode="lines+markers",
        connectgaps=False,
        line=dict(
            color=display_color,
            width=1.8,
        ),
        marker=dict(
            color=display_color,
            size=3.5,
        ),
        name="Observed four-day RQ NTL",
        legend="legend2",
        customdata=obs_short[
            ["sc_pct", "n_valid", "n_fixed"]
        ],
        hovertemplate=(
            "%{x|%d %b %Y}"
            "<br>NTL %{y:.2f}"
            "<br>SC %{customdata[0]:.0f}%"
            "<br>Pixels %{customdata[1]:.0f}/"
            "%{customdata[2]:.0f}"
            "<extra></extra>"
        ),
    ), row=2, col=1)

    # ------------------------------------------------------------
    # Left, bottom: STL components + observed − expected
    # ------------------------------------------------------------

    fig.add_trace(go.Scatter(
        x=fit.date,
        y=fit.trend_log,
        mode="lines",
        line=dict(
            color="#243B5A",
            width=2.2,
        ),
        name="Trend",
        legend="legend3",
    ), row=5, col=1, secondary_y=False)

    fig.add_trace(go.Scatter(
        x=fit.date,
        y=fit.seasonal_log,
        mode="lines",
        line=dict(
            color="#009E73",
            width=1.7,
        ),
        name="28-day component",
        legend="legend3",
    ), row=5, col=1, secondary_y=False)

    fig.add_trace(go.Scatter(
        x=fit.date,
        y=fit.residual_log,
        mode="markers",
        marker=dict(
            color="#CC79A7",
            size=4,
            opacity=.75,
        ),
        name="Residual",
        legend="legend3",
    ), row=5, col=1, secondary_y=False)

    fig.add_trace(go.Scatter(
        x=departure.date,
        y=departure.difference_ntl,
        mode="lines+markers",
        connectgaps=False,
        line=dict(
            color="#0072B2",
            width=1.9,
        ),
        marker=dict(
            color="#0072B2",
            size=3.5,
        ),
        name="Observed − expected",
        legend="legend3",
        hovertemplate=(
            "%{x|%d %b %Y}"
            "<br>Observed − expected %{y:.2f}"
            "<extra></extra>"
        ),
    ), row=5, col=1, secondary_y=True)

    fig.add_trace(go.Scatter(
        x=[plot_start, plot_end],
        y=[0, 0],
        mode="lines",
        line=dict(
            color=BASELINE_COLOR,
            width=1,
        ),
        hoverinfo="skip",
        showlegend=False,
    ), row=5, col=1, secondary_y=True)

    # ------------------------------------------------------------
    # Shared extent for BOTH maps
    # ------------------------------------------------------------

    if item.scale == "ghsl":
        x0, y0, x1, y1 = map(
            float,
            region.total_bounds,
        )

    elif item.scale == "municipality":
        selected = focused[
            focused.unit_name.eq(item.municipality)
        ]
        geometry = selected.geometry.iloc[0]
        x0, y0, x1, y1 = geometry.bounds

    else:
        lon = float(item.longitude)
        lat = float(item.latitude)
        radius = (
            .065 if item.scale == "pixel1"
            else .085
        )
        x0 = lon - radius
        x1 = lon + radius
        y0 = lat - radius
        y1 = lat + radius

    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2
    cos_lat = np.cos(np.deg2rad(cy))

    # Square in geographic distance, with modest surrounding context.
    side_lat = 1.12 * max(
        (x1 - x0) * cos_lat,
        y1 - y0,
    )
    if item.scale == "pixel1":
        side_lat = max(side_lat, .14)
    elif item.scale == "kernel5":
        side_lat = max(side_lat, .18)

    bounds = (
        cx - side_lat / (2 * cos_lat),
        cy - side_lat / 2,
        cx + side_lat / (2 * cos_lat),
        cy + side_lat / 2,
    )

    map_step = 3 if item.scale == "ghsl" else 1
    xi = np.flatnonzero(
        (a2.x.values >= bounds[0])
        & (a2.x.values <= bounds[2])
    )[::map_step]
    yi = np.flatnonzero(
        (a2.y.values >= bounds[1])
        & (a2.y.values <= bounds[3])
    )[::map_step]

    # Upper-right: GHSL.
    fig.add_trace(go.Heatmap(
        x=a2.x.values[xi],
        y=a2.y.values[yi],
        z=ghsl_grid[np.ix_(yi, xi)],
        colorscale=ghsl_scale,
        zmin=-.5,
        zmax=len(ghsl_codes) - .5,
        zsmooth=False,
        showscale=True,
        colorbar=dict(
            title="GHSL",
            x=1.01,
            y=sum(ghsl_map_domain) / 2,
            len=(ghsl_map_domain[1]
                 - ghsl_map_domain[0]) * .72,
            thickness=12,
            tickvals=list(range(len(ghsl_codes))),
            ticktext=[str(code) for code in ghsl_codes],
            tickfont=dict(size=10),
        ),
        customdata=ghsl[np.ix_(yi, xi)],
        hoverongaps=False,
        hovertemplate=(
            "%{x:.3f}°E, %{y:.3f}°N"
            "<br>GHSL %{customdata:.0f}"
            "<extra></extra>"
        ),
    ), row=1, col=2)

    # Lower-right: same extent, 60-day reference NTL.
    fig.add_trace(go.Heatmap(
        x=a2.x.values[xi],
        y=a2.y.values[yi],
        z=reference_grid[np.ix_(yi, xi)],
        colorscale="Inferno",
        zmin=0,
        zmax=5,
        zsmooth=False,
        showscale=True,
        colorbar=dict(
            title="NTL<br>nW cm⁻² sr⁻¹",
            x=1.01,
            y=sum(ntl_map_domain) / 2,
            len=(ntl_map_domain[1]
                 - ntl_map_domain[0]) * .72,
            thickness=12,
            tickfont=dict(size=10),
        ),
        hoverongaps=False,
        hovertemplate=(
            "%{x:.3f}°E, %{y:.3f}°N"
            "<br>Reference NTL %{z:.2f}"
            "<extra></extra>"
        ),
    ), row=4, col=2)

    # Borders, AOI and Haiyan path on BOTH maps.
    for map_row, axis_number in (
        (1, "2"),
        (4, "4"),
    ):
        fig.add_trace(go.Scatter(
            x=boundary_x,
            y=boundary_y,
            mode="lines",
            line=dict(
                color=(
                    "rgba(36,59,90,.45)"
                    if map_row == 1
                    else "rgba(185,200,218,.65)"
                ),
                width=.7,
            ),
            hoverinfo="skip",
            showlegend=False,
        ), row=map_row, col=2)

        if item.scale in ("pixel1", "kernel5"):
            support = fixed_profiles[pid]
            iy_support, ix_support = np.unravel_index(
                land_pixels[support],
                land.shape,
            )
            selected_x = a2.x.values[ix_support]
            selected_y = a2.y.values[iy_support]
            box = (
                float(selected_x.min() - dx / 2),
                float(selected_y.min() - dy / 2),
                float(selected_x.max() + dx / 2),
                float(selected_y.max() + dy / 2),
            )

            for color, width in (
                ("white", 5),
                ("#111827", 2.5),
            ):
                fig.add_shape(
                    type="rect",
                    xref=f"x{axis_number}",
                    yref=f"y{axis_number}",
                    x0=box[0],
                    y0=box[1],
                    x1=box[2],
                    y1=box[3],
                    line=dict(
                        color=color,
                        width=width,
                    ),
                    fillcolor="rgba(0,0,0,0)",
                )

        elif item.scale == "municipality":
            edge = geometry.boundary
            segments = (
                edge.geoms
                if hasattr(edge, "geoms")
                else [edge]
            )
            for segment in segments:
                xx, yy = segment.xy
                for color, width in (
                    ("white", 4),
                    ("#111827", 2),
                ):
                    fig.add_trace(go.Scatter(
                        x=list(xx),
                        y=list(yy),
                        mode="lines",
                        line=dict(
                            color=color,
                            width=width,
                        ),
                        hoverinfo="skip",
                        showlegend=False,
                    ), row=map_row, col=2)

        elif item.scale == "ghsl":
            class_mask = (
                (ghsl[np.ix_(yi, xi)]
                 == int(item.ghsl_code))
                & (regional_zone[np.ix_(yi, xi)] > 0)
            ).astype(float)

            fig.add_trace(go.Contour(
                x=a2.x.values[xi],
                y=a2.y.values[yi],
                z=class_mask,
                contours=dict(
                    start=.5,
                    end=.5,
                    size=1,
                    coloring="lines",
                    showlabels=False,
                ),
                line=dict(
                    color="white",
                    width=1.2,
                ),
                showscale=False,
                hoverinfo="skip",
                showlegend=False,
            ), row=map_row, col=2)

        if track_x:
            # Halo keeps the black path visible over both maps.
            for color, width in (
                ("white", 4),
                ("#111827", 2),
            ):
                fig.add_trace(go.Scatter(
                    x=track_x,
                    y=track_y,
                    mode="lines",
                    line=dict(
                        color=color,
                        width=width,
                    ),
                    hovertemplate=(
                        "Haiyan path<extra></extra>"
                    ),
                    showlegend=False,
                ), row=map_row, col=2)

    # ------------------------------------------------------------
    # Axes, aspect ratio and subplot legends
    # ------------------------------------------------------------

    for row in (1, 2, 5):
        fig.update_xaxes(
            range=[plot_start, plot_end],
            tickformat="%b %Y",
            dtick="M3",
            tickfont=dict(size=12),
            showticklabels=(row == 5),
            row=row,
            col=1,
        )

    fig.update_yaxes(
        showticklabels=False,
        showgrid=False,
        row=1,
        col=1,
    )
    fig.update_yaxes(
        title_text="NTL (nW cm⁻² sr⁻¹)",
        rangemode="tozero",
        tickfont=dict(size=12),
        row=2,
        col=1,
    )
    fig.update_yaxes(
        title_text="STL · log(1 + NTL)",
        tickfont=dict(size=12),
        row=5,
        col=1,
        secondary_y=False,
    )
    fig.update_yaxes(
        title_text="Observed − expected",
        tickfont=dict(size=12),
        row=5,
        col=1,
        secondary_y=True,
    )

    for row in (1, 4):
        fig.update_xaxes(
            range=[bounds[0], bounds[2]],
            showgrid=False,
            zeroline=False,
            nticks=3,
            tickfont=dict(size=10),
            ticksuffix="°E",
            constrain="domain",
            row=row,
            col=2,
        )
        fig.update_yaxes(
            range=[bounds[1], bounds[3]],
            showgrid=False,
            zeroline=False,
            nticks=3,
            tickfont=dict(size=10),
            ticksuffix="°N",
            scaleanchor=(
                "x2" if row == 1 else "x4"
            ),
            scaleratio=1 / cos_lat,
            constrain="domain",
            row=row,
            col=2,
        )

    for row in (1, 2, 5):
        fig.add_vline(
            x=EVENT_DATE,
            line_color=EVENT_COLOR,
            line_dash="dash",
            line_width=1.4,
            row=row,
            col=1,
        )

    style_figure(
        fig,
        width=1600,
        height=1000,
        bottom=90,
    )
    fig.update_layout(
        title=dict(
            text=item.label,
            x=.035,
            font=dict(size=21),
        ),
        margin=dict(
            l=100,
            r=105,
            t=85,
            b=85,
        ),
        legend2=dict(
            orientation="h",
            x=left_domain[0],
            y=ntl_domain[1] + .02,
            xanchor="left",
            yanchor="bottom",
            font=dict(size=16),
            bgcolor="rgba(255,255,255,.9)",
        ),
        legend3=dict(
            orientation="h",
            x=left_domain[0],
            y=components_domain[1] + .02,
            xanchor="left",
            yanchor="bottom",
            font=dict(size=16),
            bgcolor="rgba(255,255,255,.9)",
        ),
    )

    for domain, label in (
        (ghsl_map_domain, "GHSL settlement"),
        (ntl_map_domain, "Pre-Haiyan reference NTL"),
    ):
        fig.add_annotation(
            x=sum(fig.layout.xaxis2.domain) / 2,
            y=domain[1] - .012,
            xref="paper",
            yref="paper",
            text=label,
            showarrow=False,
            font=dict(
                size=15,
                color=PLOT_TEXT_COLOR,
            ),
        )

    filename = (
        "03_stl_alternate_"
        + pid.replace("::", "_")
             .replace(" ", "_")
             .replace("/", "_")
    )
    finish_figure(
        fig,
        filename,
        show=pid in show_profiles,
    )
    saved_count += 1

print(
    f"Saved {saved_count} profile-specific figures "
    f"in {FIGURE_DIR}."
)

The compact three-row map design below repairs an accidentally pasted duplicate inside its final annotation; its original traces and styling are otherwise retained.

In [ ]:
show_profiles = {
    "municipality::Tacloban",
    "centre5::Tacloban",
    "ghsl::30",
    "pixel1::Airport",
}

plot_start = dates.min()
plot_end = min(
    dates.max(),
    EVENT_DATE + pd.Timedelta(days=FORECAST_DAYS),
)

reference_grid = np.full(land.shape, np.nan, dtype=float)
reference_grid.ravel()[
    land_pixels[reference_ok]
] = ntl0[reference_ok]

dx = float(abs(a2.x.values[1] - a2.x.values[0]))
dy = float(abs(a2.y.values[1] - a2.y.values[0]))

boundary_x, boundary_y = [], []
for geometry in region.geometry:
    edge = geometry.boundary
    segments = edge.geoms if hasattr(edge, "geoms") else [edge]
    for segment in segments:
        xx, yy = segment.xy
        boundary_x.extend([*xx, None])
        boundary_y.extend([*yy, None])

sc_colorscale = [
    [0.00, "#F3F5F8"],
    [0.10, "#DCEEDD"],
    [0.50, "#75BD83"],
    [1.00, "#006B2D"],
]

saved_count = 0

for item in tqdm(
    profile_meta.loc[profile_meta.scale.ne("regional")].itertuples(),
    total=len(profile_meta.loc[profile_meta.scale.ne("regional")]),
    desc="STL figures",
):
    pid = item.profile_id

    fit = stl_components[
        stl_components.profile_id.eq(pid)
    ].sort_values("date")
    if fit.empty:
        continue

    obs = four_day[
        four_day.profile_id.eq(pid)
    ].sort_values("date")
    obs_short = obs[obs.date.le(plot_end)].copy()

    future_dates = pd.DatetimeIndex(
        forecast.loc[
            forecast.profile_id.eq(pid)
            & forecast.date.le(plot_end),
            "date",
        ].sort_values()
    )
    if len(future_dates) == 0:
        continue

    display_color = {
        "#ffff00": "#B49B00",
        "#cdf57a": "#789A36",
        "#abcd66": "#789A36",
    }.get(str(item.color).lower(), item.color)

    # ------------------------------------------------------------
    # Curved projection of the pre-event STL fit
    # ------------------------------------------------------------

    pre_dates = pd.DatetimeIndex(fit.date)
    fit_log = (
        fit.trend_log.to_numpy(dtype=float)
        + fit.seasonal_log.to_numpy(dtype=float)
    )

    fit_envelope = (
        pd.Series(fit_log)
        .rolling(
            window=STL_PERIOD_BLOCKS,
            center=True,
            min_periods=1,
        )
        .mean()
        .to_numpy()
    )

    analogue_dates = future_dates - pd.Timedelta(days=365)

    pre_day = np.asarray(
        (pre_dates - pre_dates[0])
        / pd.Timedelta(days=1),
        dtype=float,
    )
    analogue_day = np.asarray(
        (analogue_dates - pre_dates[0])
        / pd.Timedelta(days=1),
        dtype=float,
    )

    prior_year_curve = np.interp(
        analogue_day,
        pre_day,
        fit_envelope,
    )
    expected_log = (
        fit_envelope[-1]
        + prior_year_curve
        - prior_year_curve[0]
    )
    expected_ntl = np.maximum(
        0,
        np.expm1(expected_log),
    )

    baseline_departure = (
        np.log1p(
            fit.observed_ntl.to_numpy(dtype=float)
        )
        - fit_envelope
    )
    valid_departure = baseline_departure[
        np.isfinite(baseline_departure)
    ]
    departure_low, departure_high = np.quantile(
        valid_departure,
        [.10, .90],
    )

    range_low = np.maximum(
        0,
        np.expm1(expected_log + departure_low),
    )
    range_high = np.maximum(
        0,
        np.expm1(expected_log + departure_high),
    )

    # Difference must use the curved projection plotted above.
    projected = pd.DataFrame({
        "date": future_dates,
        "expected_ntl": expected_ntl,
    })
    departure = (
        obs_short.loc[
            obs_short.date.ge(EVENT_DATE),
            ["date", "ntl_rq"],
        ]
        .merge(projected, on="date", how="left")
    )
    departure["difference_ntl"] = (
        departure.ntl_rq - departure.expected_ntl
    )

    recent = obs.loc[
        obs.date.ge(
            EVENT_DATE - pd.Timedelta(days=365)
        )
        & obs.date.lt(EVENT_DATE),
        "ntl_rq",
    ].dropna()

    # ------------------------------------------------------------
    # Exactly three rows, two columns:
    # left = 70%; map spans all three rows on the right.
    # ------------------------------------------------------------

    fig = make_subplots(
        rows=3,
        cols=2,
        specs=[
            [{}, {"rowspan": 3}],
            [{}, None],
            [{"secondary_y": True}, None],
        ],
        column_widths=[.70, .30],
        row_heights=[.10, .53, .37],
        horizontal_spacing=.07,
        vertical_spacing=.075,
    )

    left_domain = fig.layout.xaxis.domain
    sc_domain = fig.layout.yaxis.domain
    ntl_domain = fig.layout.yaxis3.domain
    components_domain = fig.layout.yaxis4.domain

    # ------------------------------------------------------------
    # Row 1, left: shaded SC strip
    # ------------------------------------------------------------

    fig.add_trace(go.Heatmap(
        x=obs_short.date,
        y=["SC"],
        z=obs_short.sc_pct.to_numpy(
            dtype=float
        )[None, :],
        colorscale=sc_colorscale,
        zmin=0,
        zmax=100,
        hoverongaps=False,
        showscale=True,
        colorbar=dict(
            title="SC (%)",
            x=left_domain[1] + .008,
            y=sum(sc_domain) / 2,
            len=sc_domain[1] - sc_domain[0],
            thickness=10,
            tickvals=[0, 50, 100],
            tickfont=dict(size=10),
        ),
        hovertemplate=(
            "%{x|%d %b %Y}"
            "<br>Spatial completeness %{z:.0f}%"
            "<extra></extra>"
        ),
        name="Spatial completeness",
    ), row=1, col=1)

    # ------------------------------------------------------------
    # Row 2, left: observed NTL and curved projection
    # ------------------------------------------------------------

    fig.add_trace(go.Scatter(
        x=future_dates,
        y=range_low,
        mode="lines",
        line=dict(width=0),
        hoverinfo="skip",
        showlegend=False,
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=future_dates,
        y=range_high,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor="rgba(213,94,0,.12)",
        name="Pre-event variation",
        legend="legend2",
        hoverinfo="skip",
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=fit.date,
        y=fit.fitted_ntl,
        mode="lines",
        line=dict(
            color="#243B5A",
            width=2,
        ),
        name="Pre-event STL fit",
        legend="legend2",
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=future_dates,
        y=expected_ntl,
        mode="lines",
        line=dict(
            color="#D55E00",
            width=2.7,
        ),
        name="Projected STL-fit curve",
        legend="legend2",
    ), row=2, col=1)

    if not recent.empty:
        fig.add_trace(go.Scatter(
            x=future_dates,
            y=np.full(
                len(future_dates),
                float(recent.median()),
            ),
            mode="lines",
            line=dict(
                color="#81909F",
                width=1.5,
                dash="dot",
            ),
            name="Recent pre-event median",
            legend="legend2",
        ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=obs_short.date,
        y=obs_short.ntl_rq,
        mode="lines+markers",
        connectgaps=False,
        line=dict(
            color=display_color,
            width=1.8,
        ),
        marker=dict(
            color=display_color,
            size=3.5,
        ),
        name="Observed four-day RQ NTL",
        legend="legend2",
        customdata=obs_short[
            ["sc_pct", "n_valid", "n_fixed"]
        ],
        hovertemplate=(
            "%{x|%d %b %Y}"
            "<br>NTL %{y:.2f}"
            "<br>SC %{customdata[0]:.0f}%"
            "<br>Pixels %{customdata[1]:.0f}/"
            "%{customdata[2]:.0f}"
            "<extra></extra>"
        ),
    ), row=2, col=1)

    # ------------------------------------------------------------
    # Row 3, left: STL components and observed − expected
    # The departure uses the right-hand y-axis because its unit
    # is radiance; STL components use log(1 + NTL).
    # ------------------------------------------------------------

    fig.add_trace(go.Scatter(
        x=fit.date,
        y=fit.trend_log,
        mode="lines",
        line=dict(
            color="#243B5A",
            width=2.2,
        ),
        name="Trend",
        legend="legend3",
    ), row=3, col=1, secondary_y=False)

    fig.add_trace(go.Scatter(
        x=fit.date,
        y=fit.seasonal_log,
        mode="lines",
        line=dict(
            color="#009E73",
            width=1.7,
        ),
        name="28-day component",
        legend="legend3",
    ), row=3, col=1, secondary_y=False)

    fig.add_trace(go.Scatter(
        x=fit.date,
        y=fit.residual_log,
        mode="markers",
        marker=dict(
            color="#CC79A7",
            size=4,
            opacity=.75,
        ),
        name="Residual",
        legend="legend3",
    ), row=3, col=1, secondary_y=False)

    fig.add_trace(go.Scatter(
        x=departure.date,
        y=departure.difference_ntl,
        mode="lines+markers",
        connectgaps=False,
        line=dict(
            color="#0072B2",
            width=1.9,
        ),
        marker=dict(
            color="#0072B2",
            size=3.5,
        ),
        name="Observed − expected",
        legend="legend3",
        hovertemplate=(
            "%{x|%d %b %Y}"
            "<br>Observed − expected %{y:.2f}"
            "<extra></extra>"
        ),
    ), row=3, col=1, secondary_y=True)

    # A zero reference on the radiance-difference axis.
    fig.add_trace(go.Scatter(
        x=[plot_start, plot_end],
        y=[0, 0],
        mode="lines",
        line=dict(
            color=BASELINE_COLOR,
            width=1,
        ),
        hoverinfo="skip",
        showlegend=False,
    ), row=3, col=1, secondary_y=True)

    # ------------------------------------------------------------
    # Right column: NTL reference map, full figure height
    # ------------------------------------------------------------

    if item.scale == "ghsl":
        x0, y0, x1, y1 = map(
            float,
            region.total_bounds,
        )
        xpad = .04 * (x1 - x0)
        ypad = .04 * (y1 - y0)
        bounds = (
            x0 - xpad,
            y0 - ypad,
            x1 + xpad,
            y1 + ypad,
        )

    elif item.scale == "municipality":
        selected = focused[
            focused.unit_name.eq(item.municipality)
        ]
        geometry = selected.geometry.iloc[0]
        x0, y0, x1, y1 = geometry.bounds
        xpad = max(.025, .10 * (x1 - x0))
        ypad = max(.025, .10 * (y1 - y0))
        bounds = (
            x0 - xpad,
            y0 - ypad,
            x1 + xpad,
            y1 + ypad,
        )

    else:
        lon = float(item.longitude)
        lat = float(item.latitude)
        span_x, span_y = (
            (.055, .10)
            if item.scale == "pixel1"
            else (.075, .13)
        )
        bounds = (
            lon - span_x,
            lat - span_y,
            lon + span_x,
            lat + span_y,
        )

    map_step = 3 if item.scale == "ghsl" else 1
    xi = np.flatnonzero(
        (a2.x.values >= bounds[0])
        & (a2.x.values <= bounds[2])
    )[::map_step]
    yi = np.flatnonzero(
        (a2.y.values >= bounds[1])
        & (a2.y.values <= bounds[3])
    )[::map_step]

    fig.add_trace(go.Heatmap(
        x=a2.x.values[xi],
        y=a2.y.values[yi],
        z=reference_grid[np.ix_(yi, xi)],
        colorscale="Inferno",
        zmin=0,
        zmax=5,
        zsmooth=False,
        colorbar=dict(
            title="NTL<br>nW cm⁻² sr⁻¹",
            x=1.01,
            y=.48,
            len=.45,
            thickness=12,
            tickfont=dict(size=11),
        ),
        hoverongaps=False,
        hovertemplate=(
            "%{x:.3f}°E, %{y:.3f}°N"
            "<br>Reference NTL %{z:.2f}"
            "<extra></extra>"
        ),
        showscale=True,
        name="60-day reference NTL",
    ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=boundary_x,
        y=boundary_y,
        mode="lines",
        line=dict(
            color="rgba(185,200,218,.65)",
            width=.7,
        ),
        hoverinfo="skip",
        showlegend=False,
    ), row=1, col=2)

    if item.scale in ("pixel1", "kernel5"):
        support = fixed_profiles[pid]
        iy_support, ix_support = np.unravel_index(
            land_pixels[support],
            land.shape,
        )
        selected_x = a2.x.values[ix_support]
        selected_y = a2.y.values[iy_support]

        box = (
            float(selected_x.min() - dx / 2),
            float(selected_y.min() - dy / 2),
            float(selected_x.max() + dx / 2),
            float(selected_y.max() + dy / 2),
        )

        for color, width in (
            ("white", 5),
            ("#111827", 2.5),
        ):
            fig.add_shape(
                type="rect",
                xref="x2",
                yref="y2",
                x0=box[0],
                y0=box[1],
                x1=box[2],
                y1=box[3],
                line=dict(
                    color=color,
                    width=width,
                ),
                fillcolor="rgba(0,0,0,0)",
            )

    elif item.scale == "municipality":
        edge = geometry.boundary
        segments = (
            edge.geoms if hasattr(edge, "geoms")
            else [edge]
        )
        for segment in segments:
            xx, yy = segment.xy
            for color, width in (
                ("white", 4),
                ("#111827", 2),
            ):
                fig.add_trace(go.Scatter(
                    x=list(xx),
                    y=list(yy),
                    mode="lines",
                    line=dict(
                        color=color,
                        width=width,
                    ),
                    hoverinfo="skip",
                    showlegend=False,
                ), row=1, col=2)

    elif item.scale == "ghsl":
        class_mask = (
            (ghsl[np.ix_(yi, xi)] == int(item.ghsl_code))
            & (regional_zone[np.ix_(yi, xi)] > 0)
        ).astype(float)

        fig.add_trace(go.Contour(
            x=a2.x.values[xi],
            y=a2.y.values[yi],
            z=class_mask,
            contours=dict(
                start=.5,
                end=.5,
                size=1,
                coloring="lines",
                showlabels=False,
            ),
            line=dict(
                color="white",
                width=1.2,
            ),
            showscale=False,
            hoverinfo="skip",
            showlegend=False,
        ), row=1, col=2)

    # ------------------------------------------------------------
    # Aligned axes, panel legends, and final styling
    # ------------------------------------------------------------

    for row in (1, 2, 3):
        fig.update_xaxes(
            range=[plot_start, plot_end],
            tickformat="%b %Y",
            dtick="M3",
            tickfont=dict(size=12),
            showticklabels=(row == 3),
            row=row,
            col=1,
        )

    fig.update_yaxes(
        showticklabels=False,
        showgrid=False,
        row=1,
        col=1,
    )
    fig.update_yaxes(
        title_text="NTL (nW cm⁻² sr⁻¹)",
        rangemode="tozero",
        tickfont=dict(size=12),
        row=2,
        col=1,
    )
    fig.update_yaxes(
        title_text="STL · log(1 + NTL)",
        tickfont=dict(size=12),
        row=3,
        col=1,
        secondary_y=False,
    )
    fig.update_yaxes(
        title_text="Observed − expected",
        tickfont=dict(size=12),
        row=3,
        col=1,
        secondary_y=True,
    )

    fig.update_xaxes(
        range=[bounds[0], bounds[2]],
        showgrid=False,
        zeroline=False,
        nticks=3,
        tickfont=dict(size=11),
        ticksuffix="°E",
        row=1,
        col=2,
    )
    fig.update_yaxes(
        range=[bounds[1], bounds[3]],
        showgrid=False,
        zeroline=False,
        nticks=5,
        tickfont=dict(size=11),
        ticksuffix="°N",
        row=1,
        col=2,
    )

    for row in (1, 2, 3):
        fig.add_vline(
            x=EVENT_DATE,
            line_color=EVENT_COLOR,
            line_dash="dash",
            line_width=1.4,
            row=row,
            col=1,
        )

    style_figure(
        fig,
        width=1600,
        height=900,
        bottom=90,
    )

    fig.update_layout(
        title=dict(
            text=item.label,
            x=.035,
            font=dict(size=21),
        ),
        margin=dict(
            l=100,
            r=95,
            t=80,
            b=85,
        ),
        legend2=dict(
            orientation="h",
            x=left_domain[0],
            y=ntl_domain[1] + .025,
            xanchor="left",
            yanchor="bottom",
            font=dict(size=13),
            bgcolor="rgba(255,255,255,.9)",
        ),
        legend3=dict(
            orientation="h",
            x=left_domain[0],
            y=components_domain[1] + .025,
            xanchor="left",
            yanchor="bottom",
            font=dict(size=13),
            bgcolor="rgba(255,255,255,.9)",
        ),
    )

    fig.add_annotation(
        x=sum(fig.layout.xaxis2.domain) / 2,
        y=1.015,
        xref="paper",
        yref="paper",
        text="Pre-Haiyan reference NTL",
        showarrow=False,
        font=dict(
            size=16,
            color=PLOT_TEXT_COLOR,
        ),
    )

    filename = (
        "03_stl_compact_"
        + pid.replace("::", "_")
             .replace(" ", "_")
             .replace("/", "_")
    )
    finish_figure(
        fig,
        filename,
        show=pid in show_profiles,
    )
    saved_count += 1

print(
    f"Saved {saved_count} profile-specific figures "
    f"in {FIGURE_DIR}."
)